# V6 native-resolution detector pilot
Private checkpoint-only inference. No training or test inference.
Exact Apache-attributed V5 architecture, 1024 versus 2048, union versus mask NMS.
Fixed 16 tuning and 16 confirmation observations on separate dates.


In [ ]:
import sys,json,subprocess
from pathlib import Path
WORK=Path('/kaggle/working/v6-refiner-audit');WORK.mkdir(parents=True,exist_ok=True)
try:
    import pycocotools
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','pycocotools==2.0.8'],check=True)

sources={'solar/__init__.py': '', 'solar/v6_audit.py': '"""Overlapping-instance forensic tools. All reported PQ values are LOCAL proxies.\n\nIndependent binary masks deliberately avoid label-map overwrite of overlapping\nannotations. This implementation is not asserted equivalent to Kaggle\'s scorer.\n"""\nfrom __future__ import annotations\nimport csv\nimport json\nfrom pathlib import Path\nimport cv2\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\nfrom pycocotools import mask as coco\n\n\ndef _masks(value):\n    value = np.asarray(value)\n    if value.ndim != 3 or not np.isin(value, [0,1]).all():\n        raise ValueError(\'Expected binary masks with shape [N,H,W]\')\n    if len(value) and (value.sum(axis=(1,2)) == 0).any():\n        raise ValueError(\'Empty instances are invalid; use [0,H,W] for no detections\')\n    return value.astype(bool, copy=False)\n\n\ndef pairwise(gt, pred):\n    gt, pred = _masks(gt), _masks(pred)\n    if gt.shape[1:] != pred.shape[1:]:\n        raise ValueError(\'Mask dimensions differ\')\n    ga = gt.sum(axis=(1,2)).astype(float)\n    pa = pred.sum(axis=(1,2)).astype(float)\n    # Avoid N*M*H*W temporaries and uint8 matrix multiplication overflow.\n    intersection = np.zeros((len(gt),len(pred)),float)\n    for i,g in enumerate(gt):\n        for j,p in enumerate(pred):\n            intersection[i,j] = np.count_nonzero(g & p)\n    iou = intersection / np.maximum(ga[:,None]+pa[None,:]-intersection,1)\n    return intersection, ga, pa, iou\n\n\ndef measure(gt, pred):\n    return measure_overlaps(*pairwise(gt,pred))\n\n\ndef measure_overlaps(intersection,ga,pa,iou):\n    # Prioritize valid matches before IoU sum for overlapping annotation masks.\n    valid = iou > .5\n    rows,cols = linear_sum_assignment(-(valid*(min(iou.shape,default=0)+1)+iou))\n    matches = [(int(r),int(c)) for r,c in zip(rows,cols) if valid[r,c]]\n    values = [float(iou[r,c]) for r,c in matches]\n    tp,fp,fn = len(values), len(pa)-len(values), len(ga)-len(values)\n    denominator = tp+.5*(fp+fn)\n    total = sum(values)\n    reference_tp=int(valid.sum())\n    reference_fp=int(np.count_nonzero(~valid.any(axis=0)))\n    reference_fn=int(np.count_nonzero(~valid.any(axis=1)))\n    reference_sum=float(iou[valid].sum())\n    reference_denominator=reference_tp+.5*(reference_fp+reference_fn)\n    return {\n        \'pq_reference\':reference_sum/reference_denominator if reference_denominator else 0.,\n        \'reference_tp\':reference_tp,\'reference_fp\':reference_fp,\'reference_fn\':reference_fn,\n        \'reference_sum_iou\':reference_sum,\n        \'pq_local\':total/denominator if denominator else 1.,\n        \'sq\':total/tp if tp else (1. if not denominator else 0.),\n        \'rq\':tp/denominator if denominator else 1.,\n        \'tp\':tp,\'fp\':fp,\'fn\':fn,\'sum_iou\':total,\n        \'precision\':tp/(tp+fp) if tp+fp else (1. if not fn else 0.),\n        \'recall\':tp/(tp+fn) if tp+fn else 1.,\n        \'matched_iou\':values,\n        \'matched_dice\':[float(2*intersection[r,c]/(ga[r]+pa[c])) for r,c in matches],\n        \'likely_splits\':int(np.count_nonzero((intersection/np.maximum(ga[:,None],1) >= .1).sum(1)>=2)),\n        \'likely_merges\':int(np.count_nonzero((intersection/np.maximum(pa[None,:],1) >= .1).sum(0)>=2)),\n        \'unmatched_gt\':[i for i in range(len(ga)) if i not in {r for r,_ in matches}],\n        \'unmatched_pred\':[i for i in range(len(pa)) if i not in {c for _,c in matches}],\n    }\n\n\ndef measure_rles(gt,pred,shape=(2048,2048)):\n    """Memory-efficient equivalent for cached COCO RLE predictions."""\n    for rle in list(gt)+list(pred):\n        if tuple(rle[\'size\']) != tuple(shape): raise ValueError(\'RLE dimensions differ\')\n        if int(coco.area(rle))<=0: raise ValueError(\'Empty RLE instance\')\n    ga=np.array([float(coco.area(r)) for r in gt])\n    pa=np.array([float(coco.area(r)) for r in pred])\n    iou=coco.iou(gt,pred,[0]*len(pred)) if gt and pred else np.zeros((len(gt),len(pred)))\n    intersection=(ga[:,None]+pa[None,:])*iou/(1+iou)\n    return measure_overlaps(intersection,ga,pa,iou)\n\n\ndef partition(records, seed=42):\n    """60/20/20 date partition for NEW runs; never reuse for already trained V5.\n\n    Dates must be eight leading YYYYMMDD characters. V5 audit requires its\n    original training manifest, rather than a newly invented holdout.\n    """\n    import datetime\n    days = sorted({r[\'stem\'][:8] for r in records})\n    for day in days:\n        datetime.datetime.strptime(day,\'%Y%m%d\')\n    if len(days) < 5:\n        raise ValueError(\'Need at least five observation days for three-way evaluation\')\n    np.random.default_rng(seed).shuffle(days)\n    n = max(1,round(.2*len(days)))\n    tuning,confirmation = set(days[:n]),set(days[n:2*n])\n    return tuple([r for r in records if (r[\'stem\'][:8] in group)]\n                 for group in (set(days)-tuning-confirmation,tuning,confirmation))\n\n\ndef bootstrap(observations, seed=42, n=2000, metric="pq_local"):\n    """Macro-observation PQ; average annotators first, bootstrap observations."""\n    if not observations or n < 1 or any(not rows for rows in observations.values()):\n        raise ValueError(\'Need nonempty observation groups and positive replicates\')\n    scores = np.array([np.mean([r[metric] for r in rows]) for rows in observations.values()])\n    rng = np.random.default_rng(seed)\n    samples = [float(rng.choice(scores,len(scores),replace=True).mean()) for _ in range(n)]\n    low,high = np.quantile(samples,[.025,.975])\n    return {\'estimate\':float(scores.mean()),\'low\':float(low),\'high\':float(high),\n            \'metric\':metric,\'unit\':\'observation\',\'aggregation\':\'macro observation; annotators averaged\',\n            \'observations\':len(scores),\'replicates\':n,\'seed\':seed}\n\n\ndef export_submission(predictions, image_stems, dest):\n    """Validate the full inventory, round-trip compressed COCO RLE, then export."""\n    stems = list(image_stems)\n    if len(set(stems)) != len(stems) or set(predictions) != set(stems):\n        raise ValueError(\'Duplicate image stems or incomplete/extra image predictions\')\n    rows=[]\n    for stem in stems:\n        masks = _masks(predictions[stem])\n        if masks.shape[1:] != (2048,2048):\n            raise ValueError(\'Competition masks must be 2048 by 2048\')\n        for i,mask in enumerate(masks,1):\n            encoded = coco.encode(np.asfortranarray(mask.astype(np.uint8)))\n            if not np.array_equal(coco.decode(encoded),mask):\n                raise ValueError(\'COCO RLE round trip failed\')\n            rows.append({\'filament_id\':f\'{stem}_{i}\',\'segmentation_rle\':encoded[\'counts\'].decode(\'ascii\')})\n    if not rows or len({r[\'filament_id\'] for r in rows}) != len(rows):\n        raise ValueError(\'Empty submission or duplicate instance IDs\')\n    dest=Path(dest); dest.parent.mkdir(parents=True,exist_ok=True)\n    with dest.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,fieldnames=[\'filament_id\',\'segmentation_rle\'])\n        writer.writeheader(); writer.writerows(rows)\n    dest.with_suffix(\'.inventory.json\').write_text(json.dumps({k:len(v) for k,v in predictions.items()},indent=2))\n\n\ndef overlay(image,gt,pred,path):\n    """Two panels with individual contours: green GT; red predictions."""\n    gt,pred = _masks(gt),_masks(pred)\n    if image.shape != gt.shape[1:] or image.shape != pred.shape[1:]:\n        raise ValueError(\'Image-mask shape mismatch\')\n    panels=[]\n    for masks,color in ((gt,(0,255,0)),(pred,(0,0,255))):\n        panel=cv2.cvtColor(image,cv2.COLOR_GRAY2BGR)\n        for i,mask in enumerate(masks,1):\n            contours,_=cv2.findContours(mask.astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)\n            cv2.drawContours(panel,contours,-1,color,1)\n            y,x=np.argwhere(mask)[0]; cv2.putText(panel,str(i),(int(x),int(y)),cv2.FONT_HERSHEY_SIMPLEX,.4,color,1)\n        panels.append(panel)\n    Path(path).parent.mkdir(parents=True,exist_ok=True)\n    if not cv2.imwrite(str(path),np.concatenate(panels,axis=1)):\n        raise OSError(f\'Cannot save {path}\')\n', 'solar/v6_cached_search.py': '"""Cheap cached-mask ablation with a fixed tuning/confirmation date partition.\n\nPost-hoc confirmation is independent of this threshold search, but V5 itself\nalready used its complete old validation set for selection. No official score.\n"""\nimport numpy as np\nfrom pycocotools import mask as coco\nfrom .v6_audit import measure_rles\n\n\ndef micro(groups):\n    rows=[row for variants in groups.values() for row in variants]\n    tp,fp,fn=(sum(row[key] for row in rows) for key in [\'reference_tp\',\'reference_fp\',\'reference_fn\'])\n    denominator=tp+.5*(fp+fn)\n    return sum(row[\'reference_sum_iou\'] for row in rows)/denominator if denominator else 0.\n\n\ndef evaluate(truth,predictions,stems,area,shape):\n    return {stem:[measure_rles(gt,[r for r in predictions[stem] if coco.area(r)>=area],shape) for gt in truth[stem]] for stem in stems}\n\n\ndef paired_ci(baseline,candidate,seed=42,n=2000):\n    if set(baseline)!=set(candidate) or not baseline:raise ValueError(\'Need paired observation results\')\n    deltas=np.array([np.mean([r[\'pq_reference\'] for r in candidate[stem]])-np.mean([r[\'pq_reference\'] for r in baseline[stem]]) for stem in sorted(baseline)])\n    rng=np.random.default_rng(seed)\n    samples=[float(rng.choice(deltas,len(deltas),replace=True).mean()) for _ in range(n)]\n    low,high=np.quantile(samples,[.025,.975])\n    return {\'estimate\':float(deltas.mean()),\'low\':float(low),\'high\':float(high),\'unit\':\'physical observation\',\'replicates\':n,\'seed\':seed}\n\n\ndef search_area(truth,predictions,training_stems,thresholds=(0,32,64,128,256,512,1024,2048),seed=42,shape=(2048,2048)):\n    if set(truth)!=set(predictions) or set(truth)&set(training_stems):raise ValueError(\'Prediction inventory or physical split differs\')\n    thresholds=sorted(set(thresholds))\n    if not thresholds or thresholds[0]!=0 or any(not np.isfinite(x) or x<0 for x in thresholds):raise ValueError(\'Area grid must include zero and nonnegative finite thresholds\')\n    train_dates={stem[:8] for stem in training_stems}\n    eligible=sorted(stem for stem in truth if stem[:8] not in train_dates)\n    dates=sorted({stem[:8] for stem in eligible})\n    if len(dates)<4:return {\'status\':\'insufficient disjoint validation dates\',\'eligible_dates\':len(dates)}\n    np.random.default_rng(seed).shuffle(dates)\n    tuning_dates=set(dates[:len(dates)//2]);confirmation_dates=set(dates[len(dates)//2:])\n    tuning=sorted(stem for stem in eligible if stem[:8] in tuning_dates)\n    confirmation=sorted(stem for stem in eligible if stem[:8] in confirmation_dates)\n    trials=[]\n    for area in thresholds:\n        groups=evaluate(truth,predictions,tuning,area,shape)\n        trials.append({\'min_area\':area,\'tuning_reference_micro_pq\':micro(groups)})\n    # Lower area wins ties: preserve fine structures unless evidence favors filtering.\n    selected=max(trials,key=lambda row:(row[\'tuning_reference_micro_pq\'],-row[\'min_area\']))[\'min_area\']\n    baseline=evaluate(truth,predictions,confirmation,0,shape)\n    candidate=evaluate(truth,predictions,confirmation,selected,shape)\n    delta=paired_ci(baseline,candidate,seed)\n    base_score=micro(baseline);candidate_score=micro(candidate)\n    return {\'status\':\'completed\',\'experiment\':\'minimum native-mask area; cached plain refiner only\',\n            \'seed\':seed,\'excluded_observations_sharing_training_dates\':len(truth)-len(eligible),\n            \'tuning_observations\':len(tuning),\'confirmation_observations\':len(confirmation),\n            \'tuning_dates\':len(tuning_dates),\'confirmation_dates\':len(confirmation_dates),\n            \'tuning_date_overlap_confirmation\':len(tuning_dates&confirmation_dates),\n            \'tuning_grid\':trials,\'selected_min_area\':selected,\n            \'confirmation\':{\'baseline_pq\':base_score,\'candidate_pq\':candidate_score,\n                            \'micro_delta\':candidate_score-base_score,\'paired_macro_delta_ci\':delta},\n            \'confirmation_gain_supported\':candidate_score>base_score and delta[\'low\']>0,\n            \'eligible_for_official_submission\':False,\n            \'limitations\':[\'Final saved test predictions used flip4, whereas this cache is plain-refiner mode.\',\n                           \'A matched flip4 validation replay is required before applying to the saved test predictions.\',\n                           \'Original V5 model/strategy selection used all original validation observations.\']}\n', 'solar/v6_diagnostics.py': '"""Descriptive instance errors; bounding-box shape is not skeleton morphology."""\nimport numpy as np\nfrom pycocotools import mask as coco\n\n\ndef instance_profile(truth,predictions):\n    iou=coco.iou(truth,predictions,[0]*len(predictions)) if truth and predictions else np.zeros((len(truth),len(predictions)))\n    matched=iou>.5\n    result={}\n    for name,rles,flags in [(\'truth\',truth,matched.any(axis=1)),(\'prediction\',predictions,matched.any(axis=0))]:\n        counts={\'all\':{\'total\':len(rles),\'matched\':int(flags.sum())}}\n        for index,rle in enumerate(rles):\n            area=int(coco.area(rle));box=coco.toBbox(rle)\n            side_min=min(box[2:]);side_max=max(box[2:])\n            if area<=0 or side_min<=0:raise ValueError(\'Profiles require nonempty masks\')\n            area_bin=int(np.searchsorted([1024,4096,16384,65536],area,side=\'right\'))\n            tags=[f\'area_bin_{area_bin}\']\n            if side_max/side_min>=5:tags.append(\'elongation_ge_5\')\n            if side_max/side_min>=10:tags.append(\'elongation_ge_10\')\n            for tag in tags:\n                row=counts.setdefault(tag,{\'total\':0,\'matched\':0})\n                row[\'total\']+=1;row[\'matched\']+=int(flags[index])\n        result[name]=counts\n    best=iou.max(axis=1) if predictions else np.zeros(len(truth))\n    result[\'unmatched_truth_best_iou\']=best[~matched.any(axis=1)].tolist()\n    return result\n\n\ndef aggregate_profiles(profiles):\n    result={\'truth\':{},\'prediction\':{}}\n    for profile in profiles:\n        for kind in [\'truth\',\'prediction\']:\n            for tag,counts in profile[kind].items():\n                row=result[kind].setdefault(tag,{\'total\':0,\'matched\':0})\n                for name in row:row[name]+=counts[name]\n    for kind in [\'truth\',\'prediction\']:\n        for row in result[kind].values():row[\'matched_fraction\']=row[\'matched\']/row[\'total\'] if row[\'total\'] else None\n    best=[value for profile in profiles for value in profile[\'unmatched_truth_best_iou\']]\n    result[\'unmatched_truth_best_iou_quantiles\']=np.quantile(best,[0,.25,.5,.75,1]).tolist() if best else []\n    result[\'area_bin_edges_native_pixels\']=[0,1024,4096,16384,65536,\'infinity\']\n    result[\'interpretation\']=\'Descriptive strict-IoU>0.5 presence; annotator variants counted separately; shape uses bounding boxes, not skeletons.\'\n    return result\n', 'solar/v5_replay.py': '"""CPU-only replay of known V5 cached RLE predictions. Never loads model pickles.\n\nOnly aggregate metrics/configuration are committed; annotations, cached masks,\nlogs, and signed download URLs remain in temporary Actions storage.\n"""\nimport csv\nimport hashlib\nimport io\nimport json\nimport pickle\nfrom collections import defaultdict\nfrom pathlib import Path\nimport cv2\nimport numpy as np\nfrom pycocotools import mask as coco\nfrom solar.v6_audit import measure_rles,bootstrap\n\n\nV3_ARTIFACT_SHA256={\n    \'training_manifest.csv\':\'356bd788d677819a81922f132b0d65b6a709e773ac0f4f8f2a28a2369e014f21\',\n    \'best_detector_predictions_1024.pkl\':\'69816e2853e28f4464cf2df3cd89ffa1c4a471d2f7e68b5e91be3d5dbfc10eef\',\n    \'refined_validation_predictions_2048.pkl\':\'139c6b348eac9295404665ae36872ad3b3251b026580910b6dbb639963670c9c\'}\n\n\ndef verify_artifacts(paths,expected):\n    if set(paths)!=set(expected):raise ValueError(\'Artifact fingerprint inventory differs\')\n    actual={name:hashlib.sha256(path.read_bytes()).hexdigest() for name,path in paths.items()}\n    if actual!=expected:raise ValueError(\'Artifact fingerprint differs from preserved V5 training session v3\')\n    return actual\n\n\nclass DataOnlyUnpickler(pickle.Unpickler):\n    def find_class(self,module,name):\n        raise pickle.UnpicklingError(\'Only primitive RLE containers are accepted\')\n\n\ndef load_predictions(path):\n    if path.stat().st_size>64*1024*1024: raise ValueError(\'Prediction cache exceeds 64 MiB budget\')\n    value=DataOnlyUnpickler(io.BytesIO(path.read_bytes())).load()\n    if not isinstance(value,dict):raise ValueError(\'Expected observation-to-RLE mapping\')\n    for stem,masks in value.items():\n        if not isinstance(stem,str) or not isinstance(masks,list):raise ValueError(\'Invalid prediction mapping\')\n        for rle in masks:\n            if not isinstance(rle,dict) or set(rle)!={\'size\',\'counts\'}:raise ValueError(\'Invalid COCO RLE\')\n            if not isinstance(rle[\'counts\'],(str,bytes)) or list(rle[\'size\']) not in [[1024,1024],[2048,2048]]:raise ValueError(\'Invalid cached mask\')\n    return value\n\n\ndef aggregate(groups):\n    rows=[r for variants in groups.values() for r in variants]\n    tp,fp,fn=(sum(row[k] for row in rows) for k in [\'reference_tp\',\'reference_fp\',\'reference_fn\'])\n    denominator=tp+.5*(fp+fn)\n    local_tp,local_fp,local_fn=(sum(row[k] for row in rows) for k in [\'tp\',\'fp\',\'fn\'])\n    local_denominator=local_tp+.5*(local_fp+local_fn)\n    matched=[value for row in rows for value in row[\'matched_iou\']]\n    return {\'reference_micro_pq\':sum(row[\'reference_sum_iou\'] for row in rows)/denominator if denominator else 0.,\n            \'reference_tp\':tp,\'reference_fp\':fp,\'reference_fn\':fn,\n            \'canonical_micro_pq\':sum(row[\'sum_iou\'] for row in rows)/local_denominator if local_denominator else 1.,\n            \'canonical_tp\':local_tp,\'canonical_fp\':local_fp,\'canonical_fn\':local_fn,\n            \'canonical_sq\':sum(row[\'sum_iou\'] for row in rows)/local_tp if local_tp else 0.,\n            \'canonical_rq\':local_tp/local_denominator if local_denominator else 0.,\n            \'canonical_precision\':local_tp/(local_tp+local_fp) if local_tp+local_fp else 0.,\n            \'canonical_recall\':local_tp/(local_tp+local_fn) if local_tp+local_fn else 0.,\n            \'likely_splits\':sum(row[\'likely_splits\'] for row in rows),\n            \'likely_merges\':sum(row[\'likely_merges\'] for row in rows),\n            \'matched_iou_quantiles\':np.quantile(matched,[0,.25,.5,.75,1]).tolist() if matched else [],\n            \'reference_macro_observation_ci\':bootstrap(groups,metric=\'pq_reference\'),\n            \'physical_observations\':len(groups),\'annotation_variants\':len(rows)}\n\n\ndef replay(annotation_path,manifest_path,detector_path,refined_path):\n    fingerprints=verify_artifacts({\'training_manifest.csv\':manifest_path,\n                                   \'best_detector_predictions_1024.pkl\':detector_path,\n                                   \'refined_validation_predictions_2048.pkl\':refined_path},V3_ARTIFACT_SHA256)\n    payload=json.loads(annotation_path.read_text())\n    manifest=list(csv.DictReader(manifest_path.open()))\n    train={r[\'observation_id\'] for r in manifest if r[\'split\']==\'train\'}\n    val={r[\'observation_id\'] for r in manifest if r[\'split\']==\'val\'}\n    if not train or not val or train&val:raise ValueError(\'Invalid original split\')\n    by_image=defaultdict(list)\n    for annotation in payload[\'annotations\']:by_image[str(annotation[\'image_id\'])].append(annotation)\n    truth=defaultdict(list)\n    for image in payload[\'images\']:\n        stem=Path(image[\'file_name\']).stem\n        if stem not in val:continue\n        if (image[\'height\'],image[\'width\'])!=(2048,2048):raise ValueError(\'Unexpected annotation dimensions\')\n        rles=[]\n        for annotation in by_image[str(image[\'id\'])]:\n            parts=coco.frPyObjects(annotation[\'segmentation\'],2048,2048)\n            if parts:\n                rle=coco.merge(parts)\n                if coco.area(rle)>0:rles.append(rle)\n        truth[stem].append(rles)\n    detector=load_predictions(detector_path);refined=load_predictions(refined_path)\n    if set(detector)!=val or set(refined)!=val or set(truth)!=val:raise ValueError(\'Cached prediction/holdout inventory differs\')\n    from solar.v6_diagnostics import instance_profile,aggregate_profiles\n    from solar.v6_cached_search import paired_ci\n    results={};strategy_groups={}\n    for strategy,predictions in [(\'detector\',detector),(\'detector_refiner_plain\',refined)]:\n        groups={};profiles=[]\n        for stem,variants in truth.items():\n            native=[]\n            for rle in predictions[stem]:\n                if list(rle[\'size\'])==[1024,1024]:\n                    mask=cv2.resize(coco.decode(rle),(2048,2048),interpolation=cv2.INTER_NEAREST)\n                    rle=coco.encode(np.asfortranarray(mask))\n                native.append(rle)\n            groups[stem]=[measure_rles(gt,native) for gt in variants]\n            profiles.extend(instance_profile(gt,native) for gt in variants)\n        results[strategy]=aggregate(groups)\n        results[strategy][\'error_profile\']=aggregate_profiles(profiles)\n        strategy_groups[strategy]=groups\n    from solar.v6_cached_search import search_area\n    area_experiment=search_area(truth,refined,train)\n    return {\'artifact_sha256\':fingerprints,\'annotation_sha256\':hashlib.sha256(annotation_path.read_bytes()).hexdigest(),\'area_ablation\':area_experiment,\'artifact_kernel_version\':\'minahussain111/solar-filament-research-v5/v3\',\n            \'score_status\':\'cached LOCAL tuning-validation replay, not an official score; association to Public 0.32 not independently established\',\n            \'matching\':\'both organizer-public all-pairs and canonical one-to-one\',\n            \'split\':{\'train_observations\':len(train),\'validation_observations\':len(val),\n                     \'physical_observation_overlap\':len(train&val),\n                     \'overlapping_acquisition_dates\':len({s[:8] for s in train}&{s[:8] for s in val})},\n            \'all_annotation_results\':results,\n            \'plain_refiner_minus_detector_paired_macro_ci\':paired_ci(strategy_groups[\'detector\'],strategy_groups[\'detector_refiner_plain\']),\n            \'limitations\':[\'Same validation set was used for V5 tuning; not independent confirmation.\',\n                           \'Cached refiner predictions are plain mode only; flip4 not reconstructed.\',\n                           \'Masks have already undergone V5 overlap union; unmerged masks are unavailable.\',\n                           \'No new training, GPU inference, official submission, or score improvement.\']}\n\n\ndef main():\n    import tempfile\n    from kaggle.api.kaggle_api_extended import KaggleApi\n    report_path=Path(\'reports/V5_CACHED_REPLAY.json\')\n    report={}\n    try:\n        api=KaggleApi();api.authenticate()\n        with tempfile.TemporaryDirectory(prefix=\'v5-cached-\') as tmp:\n            root=Path(tmp)\n            api.kernels_pull(\'minahussain111/solar-filament-research-v5/v3\',str(root/\'source\'))\n            sources=list((root/\'source\').glob(\'*.ipynb\'))\n            if len(sources)!=1:raise ValueError(\'Missing training-session source\')\n            raw=sources[0].read_bytes()\n            expected_source=\'2781c89d49a31a08e4f3b96d39b439e029bc7535fc596bee3d8e074baf9a2bea\'\n            if hashlib.sha256(raw).hexdigest()!=expected_source:raise ValueError(\'Training-session source fingerprint changed\')\n            snapshot=Path(\'notebooks/baselines/v5_training_session_v3.ipynb\')\n            snapshot.parent.mkdir(parents=True,exist_ok=True)\n            if snapshot.exists() and snapshot.read_bytes()!=raw:raise ValueError(\'Refusing baseline source overwrite\')\n            snapshot.write_bytes(raw)\n            pattern=r\'(training_manifest\\.csv|best_detector_predictions_1024\\.pkl|refined_validation_predictions_2048\\.pkl|detector_thresholds\\.json|final_configuration\\.json|detector_calibration\\.csv)$\'\n            next_page=None\n            for page in range(10):\n                _,next_page=api.kernels_output(\'minahussain111/solar-filament-research-v5/v3\',str(root/\'output\'),file_pattern=pattern,page_token=next_page,page_size=100)\n                if not next_page:break\n            else:raise RuntimeError(\'Output inventory pagination budget exceeded\')\n            desired=\'MAGFiLO_1.0_Annotations_kaggle2026_train.json\'\n            annotation_name=None;next_page=None\n            for page in range(30):\n                listing=api.competition_list_files(\'filament-segmentation-2026\',page_token=next_page,page_size=100)\n                matches=[file.name for file in listing.files if Path(file.name).name==desired]\n                if len(matches)>1:raise ValueError(\'Ambiguous official training JSON\')\n                if matches:annotation_name=matches[0];break\n                next_page=getattr(listing,\'next_page_token\',None) or getattr(listing,\'nextPageToken\',None)\n                if not next_page:break\n            if not annotation_name:raise FileNotFoundError(\'Official training JSON not exposed as an individual competition file\')\n            api.competition_download_file(\'filament-segmentation-2026\',annotation_name,str(root/\'data\'),quiet=True,unzip=True)\n            def one(name):\n                found=list(root.rglob(name))\n                if len(found)!=1:raise ValueError(f\'Expected one {name}, found {len(found)}\')\n                return found[0]\n            report=replay(one(desired),one(\'training_manifest.csv\'),one(\'best_detector_predictions_1024.pkl\'),one(\'refined_validation_predictions_2048.pkl\'))\n            report[\'selected_configuration\']=json.loads(one(\'final_configuration.json\').read_text())\n            report[\'detector_thresholds\']=json.loads(one(\'detector_thresholds.json\').read_text())\n            try:\n                quota=api.quota_view().gpu_quota\n                report[\'gpu_quota\']={\'total_hours\':quota.total_time_allowed.total_seconds()/3600,\n                                     \'used_hours\':quota.time_used.total_seconds()/3600}\n            except Exception as error:\n                report[\'gpu_quota\']={\'status\':\'unavailable\',\'error_type\':type(error).__name__}\n    except Exception as error:\n        report={\'status\':\'blocked\',\'error_type\':type(error).__name__,\n                \'http_status\':getattr(getattr(error,\'response\',None),\'status_code\',None)}\n        # Controlled invariant errors only; never serialize credential-bearing SDK exceptions.\n        if type(error) in (ValueError,FileNotFoundError,RuntimeError,pickle.UnpicklingError):report[\'diagnosis\']=str(error)[:500]\n    report_path.write_text(json.dumps(report,indent=2)+\'\\n\')\n    print(\'Saved aggregate replay report; raw annotations, predictions and credentials not published.\')\n    if report.get(\'status\')==\'blocked\':raise SystemExit(1)\n\n\nif __name__==\'__main__\':main()\n', 'solar/v6_refiner_audit.py': '"""Inference-only replay of preserved Apache-attributed V5 refiner components.\n\nNo training or pretrained downloads. Only an exact archived source fingerprint\nis accepted; only named pure definitions are loaded, never notebook cells.\n"""\nimport ast\nimport csv\nimport hashlib\nimport json\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\nimport cv2\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torchvision.models import resnet18,ResNet18_Weights\nfrom pycocotools import mask as coco\nfrom .v6_audit import measure_rles,overlay\nfrom .v6_cached_search import micro,paired_ci\n\nSOURCE_SHA=\'2781c89d49a31a08e4f3b96d39b439e029bc7535fc596bee3d8e074baf9a2bea\'\nREFINER_SHA=\'0bf638645601d436f048ef238c9cd1336b4ff73184ed4553f20b11b44db09e57\'\nTHRESHOLDS=(.65,.75,.85,.9)\n\n\ndef load_components(path):\n    raw=Path(path).read_bytes()\n    if hashlib.sha256(raw).hexdigest()!=SOURCE_SHA:raise ValueError(\'Unexpected archived V5 training source\')\n    names={\'ConvBlock\',\'CropUNet\',\'square_bounds\',\'predict_refiner_probabilities\'}\n    found={}\n    for cell in json.loads(raw)[\'cells\']:\n        if cell[\'cell_type\']!=\'code\':continue\n        code=\'\'.join(cell[\'source\'])\n        if code.startswith(\'%%writefile\'):code=code.split(\'\\n\',1)[1]\n        if code.lstrip().startswith(\'!\'):continue\n        try:tree=ast.parse(code)\n        except SyntaxError:continue\n        for node in tree.body:\n            if isinstance(node,(ast.ClassDef,ast.FunctionDef)) and node.name in names:found.setdefault(node.name,node)\n    if set(found)!=names:raise ValueError(\'Missing archived model definitions\')\n    namespace={\'torch\':torch,\'nn\':nn,\'np\':np,\'resnet18\':resnet18,\'ResNet18_Weights\':ResNet18_Weights}\n    selected=[found[name] for name in [\'ConvBlock\',\'CropUNet\',\'square_bounds\',\'predict_refiner_probabilities\']]\n    exec(compile(ast.Module(body=selected,type_ignores=[]),\'archived_apache_v5_helpers\',\'exec\'),namespace)\n    return namespace\n\n\ndef postprocess(rles,scores,policy):\n    if policy not in {\'union\',\'nms\',\'none\'}:raise ValueError(\'Unknown overlap policy\')\n    scores=np.asarray(scores,dtype=float)\n    if scores.shape!=(len(rles),) or not np.isfinite(scores).all():raise ValueError(\'One finite confidence per mask is required\')\n    if not rles:return []\n    if policy==\'none\':return list(rles)\n    iou=coco.iou(rles,rles,[0]*len(rles))\n    if policy==\'nms\':\n        keep=[]\n        for index in np.argsort(-np.asarray(scores),kind=\'stable\'):\n            if all(iou[index,previous]<=.5 for previous in keep):keep.append(int(index))\n        return [rles[index] for index in keep]\n    if policy!=\'union\':raise ValueError(\'Unknown overlap policy\')\n    seen=set();result=[]\n    for start in range(len(rles)):\n        if start in seen:continue\n        stack=[start];group=[]\n        while stack:\n            index=stack.pop()\n            if index in seen:continue\n            seen.add(index);group.append(index)\n            stack.extend(np.flatnonzero(iou[index]>0).tolist())\n        result.append(coco.merge([rles[index] for index in group]))\n    return result\n\n\ndef key(mode,threshold,policy):return f\'{mode}|threshold={threshold:.2f}|{policy}\'\n\n\ndef references(path,stems):\n    payload=json.loads(Path(path).read_text());annotations=defaultdict(list);truth=defaultdict(list)\n    for ann in payload[\'annotations\']:annotations[str(ann[\'image_id\'])].append(ann)\n    for image in payload[\'images\']:\n        stem=Path(image[\'file_name\']).stem\n        if stem not in stems:continue\n        masks=[]\n        for ann in annotations[str(image[\'id\'])]:\n            parts=coco.frPyObjects(ann[\'segmentation\'],2048,2048)\n            if parts:\n                rle=coco.merge(parts)\n                if coco.area(rle)>0:masks.append(rle)\n        truth[stem].append(masks)\n    if set(truth)!=set(stems):raise ValueError(\'Ground-truth observation inventory differs\')\n    return dict(truth)\n\n\ndef evaluate(truth,predictions,stems):\n    return {stem:[measure_rles(gt,predictions[stem]) for gt in truth[stem]] for stem in stems}\n\n\n@torch.inference_mode()\ndef run(annotation,manifest,detector_path,refined_path,checkpoint,source,out,seconds=1000):\n    from .v5_replay import load_predictions,verify_artifacts,V3_ARTIFACT_SHA256\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    report={\'status\':\'running\',\'training\':False,\'test_inference\':False,\'official_submission\':False,\n            \'source_sha256\':SOURCE_SHA,\'refiner_sha256\':REFINER_SHA,\'threshold_grid\':list(THRESHOLDS)}\n    torch.manual_seed(42);np.random.seed(42)\n    try:\n        verify_artifacts({\'training_manifest.csv\':manifest,\'best_detector_predictions_1024.pkl\':detector_path,\n                          \'refined_validation_predictions_2048.pkl\':refined_path},V3_ARTIFACT_SHA256)\n        if hashlib.sha256(Path(checkpoint).read_bytes()).hexdigest()!=REFINER_SHA:raise ValueError(\'Refiner checkpoint fingerprint differs\')\n        if not torch.cuda.is_available():raise RuntimeError(\'CUDA is required for this bounded replay\')\n        helpers=load_components(source)\n        model=helpers[\'CropUNet\'](pretrained=False)\n        saved=torch.load(checkpoint,map_location=\'cpu\',weights_only=True)\n        model.load_state_dict(saved[\'model\'],strict=True);model=model.cuda().eval()\n        report.update(gpu=torch.cuda.get_device_name(0),visible_gpus=torch.cuda.device_count(),used_gpus=1,\n                      torch=str(torch.__version__),checkpoint_epoch=int(saved[\'epoch\']))\n        rows=list(csv.DictReader(Path(manifest).open()))\n        training={r[\'observation_id\'] for r in rows if r[\'split\']==\'train\'}\n        validation={r[\'observation_id\']:r for r in rows if r[\'split\']==\'val\'}\n        detector=load_predictions(Path(detector_path));cached_plain=load_predictions(Path(refined_path))\n        if set(detector)!=set(validation) or set(cached_plain)!=set(validation):raise ValueError(\'Original validation inventory differs\')\n        truth=references(annotation,set(validation))\n        strategies={key(mode,threshold,policy):{} for mode in [\'plain\',\'flip4\'] for threshold in THRESHOLDS for policy in [\'union\',\'nms\',\'none\']}\n        images=Path(annotation).parent/\'train_images\'\n        started=time.perf_counter();torch.cuda.reset_peak_memory_stats();completed=0\n        for stem,row in sorted(validation.items()):\n            if time.perf_counter()-started>seconds:raise RuntimeError(\'Inference time budget exhausted; no partial-result promotion\')\n            path=images/row[\'file_name\'];image=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE)\n            if image is None or image.shape!=(2048,2048):raise ValueError(\'Missing native validation image\')\n            tasks=[]\n            for rle in detector[stem]:\n                native=cv2.resize(coco.decode(rle),(2048,2048),interpolation=cv2.INTER_NEAREST)\n                fallback=coco.encode(np.asfortranarray(native))\n                bounds=helpers[\'square_bounds\'](native)\n                x0,y0,x1,y1=bounds\n                crop=cv2.resize(image[y0:y1,x0:x1],(256,256),interpolation=cv2.INTER_LINEAR)\n                tasks.append((bounds,crop,fallback))\n            raw={(mode,threshold):([],[]) for mode in [\'plain\',\'flip4\'] for threshold in THRESHOLDS}\n            for start in range(0,len(tasks),32):\n                batch_tasks=tasks[start:start+32]\n                batch=torch.from_numpy(np.stack([task[1] for task in batch_tasks])).float().unsqueeze(1).repeat(1,3,1,1).cuda()/255\n                for mode in [\'plain\',\'flip4\']:\n                    probs=helpers[\'predict_refiner_probabilities\'](model,batch,use_tta=(mode==\'flip4\')).cpu().numpy()\n                    for (bounds,_,fallback),probability in zip(batch_tasks,probs):\n                        x0,y0,x1,y1=bounds\n                        full_probability=cv2.resize(probability,(x1-x0,y1-y0),interpolation=cv2.INTER_LINEAR)\n                        for threshold in THRESHOLDS:\n                            mask=full_probability>=threshold\n                            if mask.any():\n                                native=np.zeros((2048,2048),np.uint8);native[y0:y1,x0:x1]=mask\n                                rle=coco.encode(np.asfortranarray(native))\n                                confidence=float(full_probability[mask].mean())\n                            else:rle=fallback;confidence=0.\n                            raw[(mode,threshold)][0].append(rle);raw[(mode,threshold)][1].append(confidence)\n            for (mode,threshold),(masks,scores) in raw.items():\n                for policy in [\'union\',\'nms\',\'none\']:strategies[key(mode,threshold,policy)][stem]=postprocess(masks,scores,policy)\n            completed+=1\n            report[\'processed_observations\']=completed\n            print(f\'Inference replay {completed}/{len(validation)}\',flush=True)\n        torch.cuda.synchronize()\n        report[\'inference_and_reconstruction_seconds\']=time.perf_counter()-started\n        report[\'end_to_end_seconds_per_observation\']=report[\'inference_and_reconstruction_seconds\']/completed\n        report[\'peak_cuda_bytes\']=torch.cuda.max_memory_allocated()\n        plain_groups=evaluate(truth,strategies[key(\'plain\',.85,\'union\')],validation)\n        cached_groups=evaluate(truth,cached_plain,validation)\n        report[\'plain_parity\']={\'cached_micro_pq\':micro(cached_groups),\'replayed_micro_pq\':micro(plain_groups)}\n        if abs(micro(plain_groups)-micro(cached_groups))>.005:raise RuntimeError(\'Original plain-refiner parity check failed\')\n        train_dates={stem[:8] for stem in training}\n        eligible=sorted(stem for stem in validation if stem[:8] not in train_dates)\n        dates=sorted({stem[:8] for stem in eligible});np.random.default_rng(42).shuffle(dates)\n        tuning_dates=set(dates[:len(dates)//2]);confirmation_dates=set(dates[len(dates)//2:])\n        tuning=sorted(stem for stem in eligible if stem[:8] in tuning_dates)\n        confirmation=sorted(stem for stem in eligible if stem[:8] in confirmation_dates)\n        if len(tuning)<1 or len(confirmation)<1:raise ValueError(\'Insufficient date-isolated holdout\')\n        baseline=key(\'flip4\',.85,\'union\')\n        trials=[{\'strategy\':name,\'tuning_reference_micro_pq\':micro(evaluate(truth,predictions,tuning))} for name,predictions in strategies.items()]\n        selected=max(trials,key=lambda row:(row[\'tuning_reference_micro_pq\'],row[\'strategy\']==baseline))[\'strategy\']\n        baseline_groups=evaluate(truth,strategies[baseline],confirmation)\n        selected_groups=evaluate(truth,strategies[selected],confirmation)\n        delta=paired_ci(baseline_groups,selected_groups)\n        from .v5_replay import aggregate\n        baseline_summary=aggregate(baseline_groups);candidate_summary=aggregate(selected_groups)\n        gain=candidate_summary[\'reference_micro_pq\']>baseline_summary[\'reference_micro_pq\'] and delta[\'low\']>0\n        report.update(status=\'completed\',baseline=baseline,selected=selected,tuning_grid=trials,\n                      tuning_observations=len(tuning),confirmation_observations=len(confirmation),\n                      excluded_training_date_observations=len(validation)-len(eligible),\n                      confirmation_baseline=baseline_summary,confirmation_candidate=candidate_summary,\n                      paired_macro_delta_ci=delta,confirmation_gain_supported=gain,\n                      score_status=\'LOCAL exploratory validation, not an official Kaggle score\',\n                      limitations=[\'Original V5 selection already used all original validation data.\',\n                                   \'Cached detector proposals were already merged; detector recall cannot be recovered here.\',\n                                   \'Final full-data model/test predictions were not run or submitted.\'])\n        # Private Kaggle outputs only: preserve chosen predictions and diagnostic panels.\n        serial={name:{stem:[{\'size\':r[\'size\'],\'counts\':r[\'counts\'].decode() if isinstance(r[\'counts\'],bytes) else r[\'counts\']} for r in strategies[name][stem]] for stem in validation} for name in {baseline,selected}}\n        (out/\'validation_predictions.json\').write_text(json.dumps(serial))\n        hardest=sorted(confirmation,key=lambda stem:sum(r[\'fp\']+r[\'fn\'] for r in baseline_groups[stem]),reverse=True)[:6]\n        for number,stem in enumerate(hardest):\n            image=cv2.imread(str(images/validation[stem][\'file_name\']),cv2.IMREAD_GRAYSCALE)\n            gt=np.stack([coco.decode(r) for r in truth[stem][0]]) if truth[stem][0] else np.zeros((0,2048,2048),bool)\n            pred=strategies[selected][stem]\n            masks=np.stack([coco.decode(r) for r in pred]) if pred else np.zeros((0,2048,2048),bool)\n            overlay(image,gt,masks,out/f\'figures/confirmation-{number:02d}.png\')\n    except Exception as error:\n        report[\'status\']=\'failed\';report[\'error_type\']=type(error).__name__\n        if type(error) in (ValueError,RuntimeError):report[\'diagnosis\']=str(error)[:500]\n        (out/\'refiner_audit.json\').write_text(json.dumps(report,indent=2)+\'\\n\')\n        raise\n    (out/\'refiner_audit.json\').write_text(json.dumps(report,indent=2)+\'\\n\')\n    return report\n', 'solar/v6_resolution_audit.py': '"""Bounded native-resolution pilot using the exact Apache-attributed V5 detector.\n\nNo training/test inference. Pilot samples are fixed before examining errors.\n"""\nimport ast\nimport csv\nimport hashlib\nimport json\nimport time\nfrom pathlib import Path\nimport cv2\nimport numpy as np\nimport torch\nfrom pycocotools import mask as coco\nfrom torchvision.models.detection import maskrcnn_resnet50_fpn_v2,MaskRCNN_ResNet50_FPN_V2_Weights\nfrom torchvision.models.detection.faster_rcnn import FastRCNNPredictor\nfrom torchvision.models.detection.mask_rcnn import MaskRCNNPredictor\nfrom torchvision.models.detection.anchor_utils import AnchorGenerator\nfrom torchvision.ops import MultiScaleRoIAlign\nfrom .v6_refiner_audit import SOURCE_SHA,REFINER_SHA,load_components,postprocess,references,evaluate\nfrom .v5_replay import aggregate,load_predictions,verify_artifacts,V3_ARTIFACT_SHA256\nfrom .v6_cached_search import micro,paired_ci\nfrom .v6_audit import measure_rles,overlay\n\nDETECTOR_SHA=\'60a8f48340cc1076710027c1b159c747035217613a61e186b8705aa6fc310a36\'\n\n\ndef original_detector(source):\n    raw=Path(source).read_bytes()\n    if hashlib.sha256(raw).hexdigest()!=SOURCE_SHA:raise ValueError(\'Archived detector source fingerprint differs\')\n    for cell in json.loads(raw)[\'cells\']:\n        code=\'\'.join(cell[\'source\'])\n        if cell[\'cell_type\']!=\'code\' or not code.startswith(\'%%writefile\'):continue\n        tree=ast.parse(code.split(\'\\n\',1)[1])\n        definitions=[node for node in tree.body if isinstance(node,ast.FunctionDef) and node.name==\'build_model\']\n        if len(definitions)==1:\n            namespace={name:globals()[name] for name in [\'maskrcnn_resnet50_fpn_v2\',\'MaskRCNN_ResNet50_FPN_V2_Weights\',\'FastRCNNPredictor\',\'MaskRCNNPredictor\',\'AnchorGenerator\',\'MultiScaleRoIAlign\']}\n            namespace[\'IMAGE_SIZE\']=1024\n            exec(compile(ast.Module(body=definitions,type_ignores=[]),\'archived_apache_v5_detector\',\'exec\'),namespace)\n            return namespace[\'build_model\'](pretrained=False)\n    raise ValueError(\'Original detector builder missing\')\n\n\ndef select_pilot_dates(training,validation,per_subset=16):\n    train_dates={s[:8] for s in training}\n    eligible=sorted(s for s in validation if s[:8] not in train_dates)\n    dates=sorted({s[:8] for s in eligible});np.random.default_rng(42).shuffle(dates)\n    tune_dates=set(dates[:len(dates)//2]);confirm_dates=set(dates[len(dates)//2:])\n    rng=np.random.default_rng(43);subsets=[]\n    for allowed in [tune_dates,confirm_dates]:\n        stems=sorted(s for s in eligible if s[:8] in allowed)\n        if len(stems)<per_subset:raise ValueError(\'Insufficient date-isolated pilot observations\')\n        subsets.append(sorted(rng.choice(stems,per_subset,replace=False).tolist()))\n    return subsets\n\n\n@torch.inference_mode()\ndef detect(model,image,resolution):\n    model.transform.min_size=(resolution,);model.transform.max_size=resolution\n    resized=cv2.resize(image,(resolution,resolution),interpolation=cv2.INTER_AREA)\n    tensor=torch.from_numpy(resized.astype(np.float32)/255).unsqueeze(0).repeat(3,1,1).cuda()\n    output=model([tensor])[0];keep=output[\'scores\']>=.6\n    scores=output[\'scores\'][keep].cpu().numpy()\n    masks=(output[\'masks\'][keep,0]>=.4).cpu().numpy().astype(np.uint8)\n    rles=[];confidence=[]\n    for mask,score in zip(masks,scores):\n        if mask.any():rles.append(coco.encode(np.asfortranarray(mask)));confidence.append(float(score))\n    return rles,confidence\n\n\n@torch.inference_mode()\ndef refine(model,helpers,image,rles):\n    tasks=[];results=[]\n    for rle in rles:\n        mask=coco.decode(rle)\n        if mask.shape!=(2048,2048):mask=cv2.resize(mask,(2048,2048),interpolation=cv2.INTER_NEAREST)\n        fallback=coco.encode(np.asfortranarray(mask.astype(np.uint8)))\n        bounds=helpers[\'square_bounds\'](mask);x0,y0,x1,y1=bounds\n        crop=cv2.resize(image[y0:y1,x0:x1],(256,256),interpolation=cv2.INTER_LINEAR)\n        tasks.append((bounds,crop,fallback))\n    for start in range(0,len(tasks),32):\n        chunk=tasks[start:start+32]\n        batch=torch.from_numpy(np.stack([r[1] for r in chunk])).float().unsqueeze(1).repeat(1,3,1,1).cuda()/255\n        probs=helpers[\'predict_refiner_probabilities\'](model,batch,use_tta=True).cpu().numpy()\n        for (bounds,_,fallback),probability in zip(chunk,probs):\n            x0,y0,x1,y1=bounds\n            mask=cv2.resize(probability,(x1-x0,y1-y0),interpolation=cv2.INTER_LINEAR)>=.85\n            if not mask.any():results.append(fallback);continue\n            native=np.zeros((2048,2048),np.uint8);native[y0:y1,x0:x1]=mask\n            results.append(coco.encode(np.asfortranarray(native)))\n    return postprocess(results,[1.]*len(results),\'union\')\n\n\n@torch.inference_mode()\ndef run(annotation,manifest,detector_predictions,plain_predictions,detector_checkpoint,refiner_checkpoint,source,out,seconds=1000):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    report={\'status\':\'running\',\'training\':False,\'test_inference\':False,\'official_submission\':False,\n            \'pilot_observations_per_subset\':16,\'refiner\':\'flip4 threshold 0.85, final overlap union\',\n            \'score_threshold\':.6,\'mask_threshold\':.4,\'source_sha256\':SOURCE_SHA}\n    try:\n        verify_artifacts({\'training_manifest.csv\':manifest,\'best_detector_predictions_1024.pkl\':detector_predictions,\n                          \'refined_validation_predictions_2048.pkl\':plain_predictions},V3_ARTIFACT_SHA256)\n        for checkpoint,expected in [(detector_checkpoint,DETECTOR_SHA),(refiner_checkpoint,REFINER_SHA)]:\n            if hashlib.sha256(Path(checkpoint).read_bytes()).hexdigest()!=expected:raise ValueError(\'Checkpoint fingerprint differs\')\n        if not torch.cuda.is_available():raise RuntimeError(\'CUDA required\')\n        torch.manual_seed(42);np.random.seed(42)\n        detector=original_detector(source)\n        saved=torch.load(detector_checkpoint,map_location=\'cpu\',weights_only=True)\n        detector.load_state_dict(saved[\'model\'],strict=True);detector=detector.cuda().eval()\n        helpers=load_components(source);refiner=helpers[\'CropUNet\'](pretrained=False)\n        refiner.load_state_dict(torch.load(refiner_checkpoint,map_location=\'cpu\',weights_only=True)[\'model\'],strict=True)\n        refiner=refiner.cuda().eval()\n        rows=list(csv.DictReader(Path(manifest).open()));training={r[\'observation_id\'] for r in rows if r[\'split\']==\'train\'}\n        validation={r[\'observation_id\']:r for r in rows if r[\'split\']==\'val\'}\n        tuning,confirmation=select_pilot_dates(training,validation)\n        stems=sorted(tuning+confirmation);truth=references(annotation,set(stems))\n        cached=load_predictions(Path(detector_predictions));images=Path(annotation).parent/\'train_images\'\n        strategies={f\'{resolution}|{policy}\':{} for resolution in [1024,2048] for policy in [\'union\',\'nms\']}\n        parity=[];started=time.perf_counter();torch.cuda.reset_peak_memory_stats()\n        report.update(gpu=torch.cuda.get_device_name(0),visible_gpus=torch.cuda.device_count(),used_gpus=1,\n                      torch=str(torch.__version__),detector_epoch=int(saved[\'epoch\']))\n        for number,stem in enumerate(stems):\n            if time.perf_counter()-started>seconds:raise RuntimeError(\'Pilot budget exhausted; refusing partial-result selection\')\n            image=cv2.imread(str(images/validation[stem][\'file_name\']),cv2.IMREAD_GRAYSCALE)\n            if image is None or image.shape!=(2048,2048):raise ValueError(\'Native training observation missing\')\n            for resolution in [1024,2048]:\n                masks,scores=detect(detector,image,resolution)\n                for policy in [\'union\',\'nms\']:\n                    proposals=postprocess(masks,scores,policy)\n                    if resolution==1024 and policy==\'union\':\n                        parity.append(measure_rles(cached[stem],proposals,shape=(1024,1024))[\'pq_local\'])\n                        if parity[-1]<.99:raise RuntimeError(\'Archived detector mask parity failed before resolution comparison\')\n                    strategies[f\'{resolution}|{policy}\'][stem]=refine(refiner,helpers,image,proposals)\n            report[\'processed_observations\']=number+1\n            print(f\'Native resolution pilot {number+1}/{len(stems)}\',flush=True)\n        torch.cuda.synchronize();report[\'inference_and_reconstruction_seconds\']=time.perf_counter()-started\n        report[\'peak_cuda_bytes\']=torch.cuda.max_memory_allocated();report[\'detector_cache_parity_min\']=min(parity)\n        trials=[{\'strategy\':name,\'tuning_micro_pq\':micro(evaluate(truth,pred,tuning))} for name,pred in strategies.items()]\n        baseline=\'1024|union\';selected=max(trials,key=lambda row:(row[\'tuning_micro_pq\'],row[\'strategy\']==baseline))[\'strategy\']\n        base=evaluate(truth,strategies[baseline],confirmation);candidate=evaluate(truth,strategies[selected],confirmation)\n        delta=paired_ci(base,candidate)\n        report.update(status=\'completed\',tuning_grid=trials,baseline=baseline,selected=selected,\n                      confirmation_baseline=aggregate(base),confirmation_candidate=aggregate(candidate),\n                      paired_macro_delta_ci=delta,confirmation_gain_supported=(micro(candidate)>micro(base) and delta[\'low\']>0),\n                      score_status=\'LOCAL small exploratory pilot, not official PQ\',\n                      limitations=[\'16 observations per subset give limited statistical power.\',\n                                   \'Original model selection reused the old validation set.\',\n                                   \'No training, alternate architecture, test inference or official submission.\'])\n        # Keep observation IDs, masks and qualitative panels only in private Kaggle output.\n        (out/\'pilot_observations.json\').write_text(json.dumps({\'tuning\':tuning,\'confirmation\':confirmation}))\n        for index,stem in enumerate(confirmation[:4]):\n            image=cv2.imread(str(images/validation[stem][\'file_name\']),cv2.IMREAD_GRAYSCALE)\n            gt=np.stack([coco.decode(r) for r in truth[stem][0]]) if truth[stem][0] else np.zeros((0,2048,2048),bool)\n            pred=strategies[selected][stem];masks=np.stack([coco.decode(r) for r in pred]) if pred else np.zeros((0,2048,2048),bool)\n            overlay(image,gt,masks,out/f\'figures/native-{index:02d}.png\')\n    except Exception as error:\n        report[\'status\']=\'failed\';report[\'error_type\']=type(error).__name__\n        if type(error) in (ValueError,RuntimeError):report[\'diagnosis\']=str(error)[:500]\n        (out/\'resolution_audit.json\').write_text(json.dumps(report,indent=2)+\'\\n\');raise\n    (out/\'resolution_audit.json\').write_text(json.dumps(report,indent=2)+\'\\n\')\n    return report\n'}
baseline='{"metadata":{"kernelspec":{"display_name":"Python 3","language":"python","name":"python3"},"language_info":{"name":"python"}},"nbformat_minor":4,"nbformat":4,"cells":[{"id":"29ccbb04-3560-4d73-b3c9-3628b4c541d7","cell_type":"markdown","source":"# Solar Filament V5 — pretrained Mask R-CNN + crop U-Net\\n\\nModified research reproduction of [Nomannic\'s Apache-2.0 Kaggle notebook](https://www.kaggle.com/code/nomannic19/cv-solar-filament-mask-r-cnn-u-net-refiner). Original author: **Nomannic**; original title: *[CV] Solar Filament | Mask R-CNN + U-Net Refiner*. Licensed Apache-2.0: https://www.apache.org/licenses/LICENSE-2.0 . See our third_party/NOTICE.md.\\n\\nChanges: remove EDA-only cells, GPU preflight, four-flip refiner TTA ablation selected on official TRAIN validation, submission RLE gate. The reference\'s two-T4 detector training, pretrained COCO weights, ResNet18 crop refiner, and validation threshold selection are retained. No test labels or metadata are used for training. Requires Kaggle Internet On + GPU T4 x2. Estimated >1h GPU; score not yet known. Preserve the 0.24 submission until this candidate is verified.\\n","metadata":{}},{"id":"de2561fe-ca73-4cbb-85fa-c98132982b77","cell_type":"code","source":"import torch, cv2, torchvision, importlib.util\\nassert torch.cuda.is_available(), \'Enable Kaggle GPU.\'\\nassert torch.cuda.device_count() >= 2, \'This two-worker DDP notebook requires Kaggle T4 x2.\'\\nassert importlib.util.find_spec(\'pycocotools\'), \'Install pycocotools.\'\\nfrom torchvision.ops import nms\\nassert nms(torch.tensor([[0.,0.,10.,10.]],device=\'cuda:0\'),torch.tensor([1.],device=\'cuda:0\'),0.5).numel()==1\\nprint(\'Preflight passed: two GPUs, torchvision CUDA ops, COCO RLE.\')\\n","metadata":{},"outputs":[],"execution_count":null},{"id":"1d7cd07b-e545-4aa4-8e6b-a020ab83617a","cell_type":"code","source":"from pathlib import Path\\nimport json, random, pickle, cv2, numpy as np, pandas as pd, torch\\nfrom pycocotools.coco import COCO\\nfrom sklearn.model_selection import train_test_split\\n\\nSEED = 2026\\nrandom.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)\\n\\nDATA_ROOT = Path(\\"/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026\\")\\nANN_PATH = DATA_ROOT / \\"train/MAGFiLO_1.0_Annotations_kaggle2026_train.json\\"\\nTRAIN_DIR, TEST_DIR = DATA_ROOT / \\"train/train_images\\", DATA_ROOT / \\"test/test_images\\"\\nWORK_DIR = Path(\\"/kaggle/working/solar_filaments_final\\")\\nWORK_DIR.mkdir(parents=True, exist_ok=True)\\n\\ncoco = COCO(str(ANN_PATH))\\nrecords = []\\nfor image_id, image_info in coco.imgs.items():\\n    annotations = coco.imgToAnns[image_id]\\n    file_name = image_info[\\"file_name\\"]\\n    records.append({\\"id\\": str(image_id), \\"file_name\\": file_name, \\"observation_id\\": Path(file_name).stem, \\"site\\": Path(file_name).stem[-2:], \\"n_filaments\\": len(annotations), \\"total_mask_area\\": sum(float(annotation[\\"area\\"]) for annotation in annotations)})\\n\\nrecords_df = pd.DataFrame(records)\\nmanifest = records_df.sort_values([\\"file_name\\", \\"n_filaments\\", \\"total_mask_area\\", \\"id\\"]).drop_duplicates(\\"file_name\\", keep=\\"last\\").reset_index(drop=True)\\nmanifest[\\"count_bin\\"] = pd.qcut(manifest[\\"n_filaments\\"], 4, labels=False, duplicates=\\"drop\\")\\nstrata = manifest[\\"site\\"] + \\"_\\" + manifest[\\"count_bin\\"].astype(str)\\ntrain_indices, val_indices = train_test_split(manifest.index, test_size=0.20, random_state=SEED, stratify=strata)\\nmanifest[\\"split\\"] = np.where(manifest.index.isin(val_indices), \\"val\\", \\"train\\")\\nmanifest.to_csv(WORK_DIR / \\"training_manifest.csv\\", index=False)\\n\\nprint(f\\"Selected physical observations: {len(manifest)}\\")\\nprint(f\\"Training observations:          {(manifest.split == \'train\').sum()}\\")\\nprint(f\\"Validation observations:        {(manifest.split == \'val\').sum()}\\")\\ndisplay(pd.crosstab(manifest.site, manifest.split))","metadata":{},"outputs":[],"execution_count":null},{"id":"f8a09446-de41-44a4-bc16-a8a197559285","cell_type":"code","source":"%%writefile /kaggle/working/solar_filaments_final/train_detector.py\\nimport os, argparse, random, time\\nfrom pathlib import Path\\nimport cv2, numpy as np, pandas as pd, torch, torch.distributed as dist\\nfrom pycocotools.coco import COCO\\nfrom pycocotools import mask as mask_utils\\nfrom scipy.optimize import linear_sum_assignment\\nfrom torch.utils.data import Dataset, DataLoader\\nfrom torch.utils.data.distributed import DistributedSampler\\nfrom torch.nn.parallel import DistributedDataParallel as DDP\\nfrom torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights\\nfrom torchvision.models.detection.faster_rcnn import FastRCNNPredictor\\nfrom torchvision.models.detection.mask_rcnn import MaskRCNNPredictor\\nfrom torchvision.models.detection.anchor_utils import AnchorGenerator\\nfrom torchvision.ops import MultiScaleRoIAlign\\nfrom tqdm.auto import tqdm\\n\\nROOT = Path(\\"/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026\\")\\nANN_PATH, IMAGE_DIR = ROOT / \\"train/MAGFiLO_1.0_Annotations_kaggle2026_train.json\\", ROOT / \\"train/train_images\\"\\nWORK_DIR = Path(\\"/kaggle/working/solar_filaments_final\\")\\nMANIFEST_PATH = WORK_DIR / \\"training_manifest.csv\\"\\nIMAGE_SIZE, SCORE_THRESHOLD, MASK_THRESHOLD, SEED = 1024, 0.60, 0.50, 2026\\ncv2.setNumThreads(0)\\n\\ndef seed_everything(seed):\\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)\\n\\ndef seed_worker(worker_id):\\n    seed = torch.initial_seed() % 2**32\\n    random.seed(seed); np.random.seed(seed)\\n\\ndef masks_to_boxes(masks):\\n    boxes = []\\n    for mask in masks:\\n        ys, xs = np.where(mask)\\n        boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1])\\n    return np.asarray(boxes, dtype=np.float32).reshape(-1, 4)\\n\\ndef encode_mask(mask):\\n    return mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))\\n\\ndef merge_overlaps(masks):\\n    masks = [np.asarray(mask, dtype=bool) for mask in masks if np.any(mask)]\\n    if len(masks) < 2:\\n        return masks\\n    rles = [encode_mask(mask) for mask in masks]\\n    adjacency = mask_utils.iou(rles, rles, [0] * len(rles)) > 0\\n    groups, seen = [], set()\\n    for start in range(len(masks)):\\n        if start in seen:\\n            continue\\n        stack, group = [start], []\\n        while stack:\\n            index = stack.pop()\\n            if index in seen:\\n                continue\\n            seen.add(index); group.append(index)\\n            stack.extend(np.flatnonzero(adjacency[index]).tolist())\\n        groups.append(group)\\n    return [np.logical_or.reduce([masks[index] for index in group]) for group in groups]\\n\\ndef instance_metrics(pred_masks, true_masks):\\n    p, t = len(pred_masks), len(true_masks)\\n    if p and t:\\n        iou = mask_utils.iou([encode_mask(mask) for mask in pred_masks], [encode_mask(mask) for mask in true_masks], [0] * t)\\n        dice = 2 * iou / (1 + iou + 1e-7)\\n        rows, cols = linear_sum_assignment(-dice)\\n        mean_dice = float(dice[rows, cols].sum() / max(p, t))\\n        tp = int((iou[rows, cols] >= 0.5).sum())\\n    else:\\n        mean_dice, tp = float(p == 0 and t == 0), 0\\n\\n    if p or t:\\n        shape = pred_masks[0].shape if p else true_masks[0].shape\\n        pred_union = np.logical_or.reduce(pred_masks) if p else np.zeros(shape, bool)\\n        true_union = np.logical_or.reduce(true_masks) if t else np.zeros(shape, bool)\\n        intersection = np.logical_and(pred_union, true_union).sum()\\n        union_dice = float(2 * intersection / (pred_union.sum() + true_union.sum() + 1e-7))\\n    else:\\n        union_dice = 1.0\\n\\n    precision, recall = tp / p if p else 0.0, tp / t if t else 0.0\\n    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\\n    return {\\"mean_dice\\": mean_dice, \\"union_dice\\": union_dice, \\"f1_50\\": f1, \\"count_error\\": abs(p - t), \\"predictions\\": p}\\n\\nclass FilamentDataset(Dataset):\\n    def __init__(self, records, augment=False):\\n        self.records, self.augment = records.reset_index(drop=True), augment\\n        self.coco = COCO(str(ANN_PATH))\\n\\n    def __len__(self):\\n        return len(self.records)\\n\\n    @staticmethod\\n    def photometric(image):\\n        valid = image > 0.01\\n\\n        if np.random.rand() < 0.70:\\n            image[valid] = image[valid] ** np.random.uniform(0.80, 1.25)\\n\\n        if np.random.rand() < 0.70:\\n            mean = image[valid].mean()\\n            contrast, gain = np.random.uniform(0.85, 1.15), np.random.uniform(0.90, 1.10)\\n            image[valid] = ((image[valid] - mean) * contrast + mean) * gain\\n\\n        if np.random.rand() < 0.15:\\n            blurred = cv2.GaussianBlur(image, (0, 0), np.random.uniform(0.30, 0.80))\\n            image[valid] = blurred[valid]\\n\\n        if np.random.rand() < 0.25:\\n            image[valid] += np.random.normal(0, np.random.uniform(0.003, 0.015), valid.sum())\\n\\n        return np.clip(image, 0, 1)\\n\\n    def __getitem__(self, index):\\n        row = self.records.iloc[index]\\n        image_id = str(row[\\"id\\"])\\n        image = cv2.imread(str(IMAGE_DIR / row[\\"file_name\\"]), cv2.IMREAD_GRAYSCALE)\\n        image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA).astype(np.float32) / 255\\n\\n        masks = []\\n        for annotation in self.coco.imgToAnns[image_id]:\\n            mask = cv2.resize(self.coco.annToMask(annotation), (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_NEAREST).astype(bool)\\n            if np.any(mask):\\n                masks.append(mask)\\n        masks = np.stack(masks) if masks else np.zeros((0, IMAGE_SIZE, IMAGE_SIZE), bool)\\n\\n        if self.augment:\\n            rotations = np.random.randint(4)\\n            image = np.rot90(image, rotations)\\n            masks = np.rot90(masks, rotations, axes=(1, 2))\\n\\n            if np.random.rand() < 0.5:\\n                image = np.flip(image, axis=1)\\n                masks = np.flip(masks, axis=2)\\n\\n            image = self.photometric(image)\\n\\n        image, masks = np.ascontiguousarray(image), np.ascontiguousarray(masks)\\n        boxes = masks_to_boxes(masks)\\n        target = {\\"boxes\\": torch.as_tensor(boxes, dtype=torch.float32), \\"labels\\": torch.ones(len(masks), dtype=torch.int64), \\"masks\\": torch.as_tensor(masks, dtype=torch.uint8), \\"image_id\\": torch.tensor([index]), \\"area\\": torch.as_tensor(masks.sum((1, 2)), dtype=torch.float32), \\"iscrowd\\": torch.zeros(len(masks), dtype=torch.int64)}\\n        image = torch.from_numpy(image).unsqueeze(0).repeat(3, 1, 1)\\n        return image, target, str(row[\\"observation_id\\"])\\n\\ndef collate_fn(batch):\\n    return tuple(zip(*batch))\\n\\ndef build_model(pretrained=True):\\n    weights = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT if pretrained else None\\n    model = maskrcnn_resnet50_fpn_v2(weights=weights, weights_backbone=None)\\n\\n    box_features = model.roi_heads.box_predictor.cls_score.in_features\\n    mask_features = model.roi_heads.mask_predictor.conv5_mask.in_channels\\n    hidden = model.roi_heads.mask_predictor.conv5_mask.out_channels\\n    model.roi_heads.box_predictor = FastRCNNPredictor(box_features, 2)\\n    model.roi_heads.mask_predictor = MaskRCNNPredictor(mask_features, hidden, 2)\\n\\n    model.rpn.anchor_generator = AnchorGenerator(sizes=((16,), (32,), (64,), (128,), (256,)), aspect_ratios=((0.5, 1.0, 2.0),) * 5)\\n    model.roi_heads.mask_roi_pool = MultiScaleRoIAlign(featmap_names=[\\"0\\", \\"1\\", \\"2\\", \\"3\\"], output_size=28, sampling_ratio=2)\\n    model.transform.min_size, model.transform.max_size = (IMAGE_SIZE,), IMAGE_SIZE\\n    return model\\n\\n@torch.inference_mode()\\ndef evaluate(model, loader, device):\\n    model.eval()\\n    values = []\\n\\n    for images, targets, _ in tqdm(loader, desc=\\"Validation\\", leave=False):\\n        output = model([images[0].to(device, non_blocking=True)])[0]\\n        keep = output[\\"scores\\"] >= SCORE_THRESHOLD\\n        pred_masks = (output[\\"masks\\"][keep, 0] >= MASK_THRESHOLD).byte().cpu().numpy().astype(bool)\\n        pred_masks = merge_overlaps(pred_masks)\\n        true_masks = targets[0][\\"masks\\"].numpy().astype(bool)\\n        values.append(instance_metrics(pred_masks, true_masks))\\n\\n    return pd.DataFrame(values).mean().to_dict()\\n\\ndef main():\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\"--mode\\", choices=[\\"validation\\", \\"final\\"], default=\\"validation\\")\\n    parser.add_argument(\\"--epochs\\", type=int, default=10)\\n    args = parser.parse_args()\\n\\n    local_rank = int(os.environ[\\"LOCAL_RANK\\"])\\n    torch.cuda.set_device(local_rank)\\n    device = torch.device(\\"cuda\\", local_rank)\\n    dist.init_process_group(\\"nccl\\")\\n    rank = dist.get_rank()\\n    seed_everything(SEED + rank)\\n    torch.backends.cudnn.benchmark = True\\n    torch.set_float32_matmul_precision(\\"high\\")\\n\\n    manifest = pd.read_csv(MANIFEST_PATH, dtype={\\"id\\": str})\\n    train_records = manifest if args.mode == \\"final\\" else manifest[manifest[\\"split\\"] == \\"train\\"]\\n    val_records = manifest[manifest[\\"split\\"] == \\"val\\"]\\n\\n    train_dataset = FilamentDataset(train_records, augment=True)\\n    sampler = DistributedSampler(train_dataset, shuffle=True, seed=SEED)\\n    train_loader = DataLoader(train_dataset, batch_size=1, sampler=sampler, num_workers=2, pin_memory=True, persistent_workers=True, collate_fn=collate_fn, worker_init_fn=seed_worker)\\n\\n    val_loader = None\\n    if rank == 0 and args.mode == \\"validation\\":\\n        val_dataset = FilamentDataset(val_records, augment=False)\\n        val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True, collate_fn=collate_fn, worker_init_fn=seed_worker)\\n\\n    model = DDP(build_model(pretrained=True).to(device), device_ids=[local_rank], broadcast_buffers=False)\\n    optimizer = torch.optim.SGD([parameter for parameter in model.parameters() if parameter.requires_grad], lr=0.005, momentum=0.9, weight_decay=1e-4)\\n    milestones = sorted(set([max(1, round(args.epochs * 0.70)), max(2, round(args.epochs * 0.90))]))\\n    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=milestones, gamma=0.1)\\n    scaler = torch.amp.GradScaler(\\"cuda\\")\\n    history, best_dice = [], -1.0\\n\\n    for epoch in range(1, args.epochs + 1):\\n        sampler.set_epoch(epoch)\\n        model.train()\\n        totals = torch.zeros(2, device=device)\\n        started = time.perf_counter()\\n\\n        progress = tqdm(train_loader, desc=f\\"Epoch {epoch:02d}/{args.epochs}\\", disable=rank != 0, leave=False)\\n        for images, targets, _ in progress:\\n            images = [image.to(device, non_blocking=True) for image in images]\\n            targets = [{key: value.to(device, non_blocking=True) for key, value in target.items()} for target in targets]\\n            optimizer.zero_grad(set_to_none=True)\\n\\n            with torch.amp.autocast(\\"cuda\\", dtype=torch.float16):\\n                losses = model(images, targets)\\n                loss = sum(losses.values())\\n\\n            scaler.scale(loss).backward()\\n            scaler.step(optimizer)\\n            scaler.update()\\n            totals[0] += loss.detach()\\n            totals[1] += 1\\n\\n        dist.all_reduce(totals)\\n        train_loss = (totals[0] / totals[1]).item()\\n        metrics = {}\\n\\n        if rank == 0 and args.mode == \\"validation\\":\\n            metrics = evaluate(model.module, val_loader, device)\\n\\n        dist.barrier(device_ids=[local_rank])\\n        current_lr = optimizer.param_groups[0][\\"lr\\"]\\n\\n        if rank == 0:\\n            row = {\\"epoch\\": epoch, \\"train_loss\\": train_loss, **metrics, \\"lr\\": current_lr, \\"minutes\\": (time.perf_counter() - started) / 60}\\n            history.append(row)\\n\\n            if args.mode == \\"validation\\" and metrics[\\"mean_dice\\"] > best_dice:\\n                best_dice = metrics[\\"mean_dice\\"]\\n                torch.save({\\"epoch\\": epoch, \\"model\\": model.module.state_dict()}, WORK_DIR / \\"best_detector.pt\\")\\n\\n            metric_text = f\\" | Dice {metrics[\'mean_dice\']:.4f} | Union {metrics[\'union_dice\']:.4f} | F1 {metrics[\'f1_50\']:.4f} | Count {metrics[\'count_error\']:.2f}\\" if metrics else \\"\\"\\n            print(f\\"Epoch {epoch:02d}/{args.epochs} | Loss {train_loss:.4f}{metric_text} | LR {current_lr:.6f} | {row[\'minutes\']:.1f} min\\", flush=True)\\n\\n        scheduler.step()\\n        dist.barrier(device_ids=[local_rank])\\n\\n    if rank == 0:\\n        checkpoint_name = \\"last_detector.pt\\" if args.mode == \\"validation\\" else \\"final_detector.pt\\"\\n        torch.save({\\"epoch\\": args.epochs, \\"model\\": model.module.state_dict()}, WORK_DIR / checkpoint_name)\\n        pd.DataFrame(history).to_csv(WORK_DIR / f\\"{args.mode}_detector_history.csv\\", index=False)\\n\\n    dist.destroy_process_group()\\n\\nif __name__ == \\"__main__\\":\\n    main()","metadata":{},"outputs":[],"execution_count":null},{"id":"7afb6703-618a-48cd-94cb-82554add2873","cell_type":"code","source":"from torchvision.models.detection import MaskRCNN_ResNet50_FPN_V2_Weights\\nfrom torchvision.models import ResNet18_Weights\\n\\n_ = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT.get_state_dict(progress=True, check_hash=True)\\n_ = ResNet18_Weights.DEFAULT.get_state_dict(progress=True, check_hash=True)\\ndel _\\nprint(\\"Pretrained weights are cached.\\")","metadata":{},"outputs":[],"execution_count":null},{"id":"d9616170-eda2-494d-98cc-24dafa41c03e","cell_type":"code","source":"!torchrun --standalone --nproc_per_node=2 /kaggle/working/solar_filaments_final/train_detector.py --mode validation --epochs 10","metadata":{},"outputs":[],"execution_count":null},{"id":"35013d2d-387c-40df-8cdb-af6c8f45c899","cell_type":"code","source":"import sys, json, pickle, importlib\\nfrom pathlib import Path\\nimport cv2, numpy as np, pandas as pd, torch\\nfrom pycocotools import mask as mask_utils\\nfrom scipy.optimize import linear_sum_assignment\\nfrom torch.utils.data import DataLoader\\nfrom tqdm.auto import tqdm\\n\\nsys.path.insert(0, str(WORK_DIR))\\nimport train_detector as detector\\nimportlib.reload(detector)\\n\\ndef encode_mask(mask):\\n    return mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))\\n\\n_reference_cache = {}\\ndef annotation_rles(record_id, size):\\n    key = (str(record_id), size)\\n\\n    if key not in _reference_cache:\\n        masks = []\\n        for annotation in coco.imgToAnns[str(record_id)]:\\n            mask = coco.annToMask(annotation)\\n\\n            if mask.shape != (size, size):\\n                mask = cv2.resize(mask, (size, size), interpolation=cv2.INTER_NEAREST)\\n\\n            if np.any(mask):\\n                masks.append(encode_mask(mask))\\n\\n        _reference_cache[key] = masks\\n\\n    return _reference_cache[key]\\n\\ndef build_reference_maps(rows, size):\\n    selected, all_references = {}, {}\\n    record_ids = records_df.groupby(\\"file_name\\")[\\"id\\"].apply(list).to_dict()\\n\\n    for row in tqdm(rows.itertuples(index=False), total=len(rows), desc=f\\"Building {size}px references\\"):\\n        observation_id = str(row.observation_id)\\n        selected[observation_id] = [annotation_rles(str(row.id), size)]\\n        all_references[observation_id] = [annotation_rles(str(record_id), size) for record_id in record_ids[row.file_name]]\\n\\n    return selected, all_references\\n\\ndef score_rles(predictions, targets):\\n    p, t = len(predictions), len(targets)\\n\\n    if p and t:\\n        iou = mask_utils.iou(predictions, targets, [0] * t)\\n        dice = 2 * iou / (1 + iou + 1e-7)\\n        rows, cols = linear_sum_assignment(-dice)\\n        matched_dice, matched_iou = dice[rows, cols], iou[rows, cols]\\n        mean_dice = float(matched_dice.sum() / max(p, t))\\n        mean_iou = float(matched_iou.sum() / max(p, t))\\n        tp = int((matched_iou >= 0.5).sum())\\n    else:\\n        mean_dice = mean_iou = float(p == 0 and t == 0)\\n        tp = 0\\n\\n    if p and t:\\n        pred_union, target_union = mask_utils.merge(predictions), mask_utils.merge(targets)\\n        intersection = float(mask_utils.area(mask_utils.merge([pred_union, target_union], intersect=True)))\\n        union_dice = 2 * intersection / (float(mask_utils.area(pred_union)) + float(mask_utils.area(target_union)) + 1e-7)\\n    else:\\n        union_dice = float(p == 0 and t == 0)\\n\\n    precision, recall = tp / p if p else 0.0, tp / t if t else 0.0\\n    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\\n    return {\\"mean_dice\\": mean_dice, \\"mean_iou\\": mean_iou, \\"union_dice\\": union_dice, \\"f1_50\\": f1, \\"count_error\\": abs(p - t), \\"predictions\\": p}\\n\\ndef evaluate_prediction_map(predictions, references):\\n    rows = []\\n\\n    for observation_id, annotation_sets in references.items():\\n        reference_scores = [score_rles(predictions.get(observation_id, []), target) for target in annotation_sets]\\n        rows.append(pd.DataFrame(reference_scores).mean().to_dict())\\n\\n    return pd.DataFrame(rows).mean().to_dict()\\n\\nval_rows = manifest[manifest[\\"split\\"] == \\"val\\"].reset_index(drop=True)\\nselected_refs_1024, all_refs_1024 = build_reference_maps(val_rows, 1024)\\n\\nval_dataset = detector.FilamentDataset(val_rows, augment=False)\\nval_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True, collate_fn=detector.collate_fn, worker_init_fn=detector.seed_worker)\\n\\ndevice = torch.device(\\"cuda:0\\")\\ndetector_model = detector.build_model(pretrained=False).to(device)\\ncheckpoint = torch.load(WORK_DIR / \\"best_detector.pt\\", map_location=device, weights_only=False)\\ndetector_model.load_state_dict(checkpoint[\\"model\\"])\\ndetector_model.eval()\\n\\nscore_grid, mask_grid = [0.40, 0.50, 0.60, 0.70, 0.80], [0.40, 0.50, 0.60]\\nconfigurations = [(score, mask) for score in score_grid for mask in mask_grid]\\nprediction_grids = {configuration: {} for configuration in configurations}\\n\\nwith torch.inference_mode():\\n    for images, _, observation_ids in tqdm(val_loader, desc=\\"Caching detector predictions\\"):\\n        output = detector_model([images[0].to(device, non_blocking=True)])[0]\\n        keep = output[\\"scores\\"] >= min(score_grid)\\n        scores = output[\\"scores\\"][keep].cpu().numpy()\\n        probabilities = output[\\"masks\\"][keep, 0].cpu().numpy()\\n        observation_id = observation_ids[0]\\n\\n        for mask_threshold in mask_grid:\\n            binary_masks = probabilities >= mask_threshold\\n\\n            for score_threshold in score_grid:\\n                masks = detector.merge_overlaps(binary_masks[scores >= score_threshold])\\n                prediction_grids[(score_threshold, mask_threshold)][observation_id] = [encode_mask(mask) for mask in masks]\\n\\ncalibration_rows = []\\nfor configuration in tqdm(configurations, desc=\\"One-time calibration\\"):\\n    selected_metrics = evaluate_prediction_map(prediction_grids[configuration], selected_refs_1024)\\n    all_metrics = evaluate_prediction_map(prediction_grids[configuration], all_refs_1024)\\n    calibration_rows.append({\\"score_threshold\\": configuration[0], \\"mask_threshold\\": configuration[1], \\"selected_dice\\": selected_metrics[\\"mean_dice\\"], \\"all_dice\\": all_metrics[\\"mean_dice\\"], \\"all_union_dice\\": all_metrics[\\"union_dice\\"], \\"all_f1_50\\": all_metrics[\\"f1_50\\"], \\"count_error\\": all_metrics[\\"count_error\\"], \\"predictions\\": all_metrics[\\"predictions\\"]})\\n\\ncalibration = pd.DataFrame(calibration_rows)\\ncalibration[\\"selection_score\\"] = (calibration[\\"selected_dice\\"] + calibration[\\"all_dice\\"]) / 2\\nbest_row = calibration.sort_values([\\"selection_score\\", \\"all_f1_50\\"], ascending=False).iloc[0]\\nbest_configuration = (float(best_row[\\"score_threshold\\"]), float(best_row[\\"mask_threshold\\"]))\\nbest_detector_predictions = prediction_grids[best_configuration]\\n\\ncalibration.to_csv(WORK_DIR / \\"detector_calibration.csv\\", index=False)\\nwith open(WORK_DIR / \\"best_detector_predictions_1024.pkl\\", \\"wb\\") as file:\\n    pickle.dump(best_detector_predictions, file)\\nwith open(WORK_DIR / \\"detector_thresholds.json\\", \\"w\\") as file:\\n    json.dump({\\"score_threshold\\": best_configuration[0], \\"mask_threshold\\": best_configuration[1]}, file, indent=2)\\n\\ndisplay(calibration.sort_values(\\"selection_score\\", ascending=False).head(10).round(3))\\nprint(f\\"Best detector thresholds: score={best_configuration[0]:.2f}, mask={best_configuration[1]:.2f}\\")\\n\\ndel detector_model\\ntorch.cuda.empty_cache()","metadata":{},"outputs":[],"execution_count":null},{"id":"e88e9050-45c5-4f90-861d-a9433ce495e0","cell_type":"code","source":"from tqdm.auto import tqdm\\n\\ndef square_bounds(mask, context=1.8, minimum=96):\\n    height, width = mask.shape\\n    ys, xs = np.where(mask)\\n    center_x, center_y = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2\\n    side = int(np.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))\\n    side = min(max(side, minimum), min(height, width))\\n    x0, y0 = int(round(center_x - side / 2)), int(round(center_y - side / 2))\\n    x1, y1 = x0 + side, y0 + side\\n\\n    if x0 < 0:\\n        x1 -= x0; x0 = 0\\n    if y0 < 0:\\n        y1 -= y0; y0 = 0\\n    if x1 > width:\\n        x0 -= x1 - width; x1 = width\\n    if y1 > height:\\n        y0 -= y1 - height; y1 = height\\n\\n    return int(x0), int(y0), int(x1), int(y1)\\n\\ndef build_crop_cache(rows, prefix):\\n    total = sum(len(coco.imgToAnns[str(record_id)]) for record_id in rows[\\"id\\"])\\n    image_path, mask_path = WORK_DIR / f\\"{prefix}_crop_images.npy\\", WORK_DIR / f\\"{prefix}_crop_masks.npy\\"\\n    images = np.lib.format.open_memmap(image_path, mode=\\"w+\\", dtype=np.uint8, shape=(total, 256, 256))\\n    masks = np.lib.format.open_memmap(mask_path, mode=\\"w+\\", dtype=np.uint8, shape=(total, 256, 256))\\n    position = 0\\n\\n    for row in tqdm(rows.itertuples(index=False), total=len(rows), desc=f\\"Caching {prefix} crops\\"):\\n        image = cv2.imread(str(TRAIN_DIR / row.file_name), cv2.IMREAD_GRAYSCALE)\\n\\n        for annotation in coco.imgToAnns[str(row.id)]:\\n            mask = coco.annToMask(annotation).astype(np.uint8)\\n            x0, y0, x1, y1 = square_bounds(mask)\\n            images[position] = cv2.resize(image[y0:y1, x0:x1], (256, 256), interpolation=cv2.INTER_LINEAR)\\n            masks[position] = cv2.resize(mask[y0:y1, x0:x1], (256, 256), interpolation=cv2.INTER_NEAREST)\\n            position += 1\\n\\n    images.flush(); masks.flush()\\n    return image_path, mask_path, total\\n\\ntrain_rows = manifest[manifest[\\"split\\"] == \\"train\\"].reset_index(drop=True)\\nval_rows = manifest[manifest[\\"split\\"] == \\"val\\"].reset_index(drop=True)\\n\\ntrain_crop_images, train_crop_masks, train_crop_count = build_crop_cache(train_rows, \\"train\\")\\nval_crop_images, val_crop_masks, val_crop_count = build_crop_cache(val_rows, \\"val\\")\\n\\nprint(f\\"Training crops:   {train_crop_count:,}\\")\\nprint(f\\"Validation crops: {val_crop_count:,}\\")\\nprint(f\\"Cache size: {sum(path.stat().st_size for path in [train_crop_images, train_crop_masks, val_crop_images, val_crop_masks]) / 2**20:.1f} MB\\")","metadata":{},"outputs":[],"execution_count":null},{"id":"94b2d7a8-c3eb-4bf9-9539-2e92dffe9abb","cell_type":"code","source":"import random\\nfrom torch import nn\\nfrom torch.utils.data import Dataset, DataLoader, ConcatDataset\\nfrom torchvision.models import resnet18, ResNet18_Weights\\n\\nclass CropDataset(Dataset):\\n    def __init__(self, image_path, mask_path, augment=False):\\n        self.images, self.masks = np.load(image_path, mmap_mode=\\"r\\"), np.load(mask_path, mmap_mode=\\"r\\")\\n        self.augment = augment\\n\\n    def __len__(self):\\n        return len(self.images)\\n\\n    def __getitem__(self, index):\\n        image = torch.from_numpy(np.asarray(self.images[index]).copy()).float() / 255\\n        mask = torch.from_numpy(np.asarray(self.masks[index]).copy()).float().unsqueeze(0)\\n\\n        if self.augment:\\n            rotations = random.randrange(4)\\n            image, mask = torch.rot90(image, rotations, (0, 1)), torch.rot90(mask, rotations, (1, 2))\\n\\n            if random.random() < 0.5:\\n                image, mask = torch.flip(image, (1,)), torch.flip(mask, (2,))\\n\\n            if random.random() < 0.5:\\n                image = image ** random.uniform(0.85, 1.20)\\n\\n            if random.random() < 0.5:\\n                image = torch.clamp(image * random.uniform(0.90, 1.10), 0, 1)\\n\\n        return image.unsqueeze(0).repeat(3, 1, 1), mask\\n\\nclass ConvBlock(nn.Sequential):\\n    def __init__(self, in_channels, out_channels):\\n        super().__init__(nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True), nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))\\n\\nclass CropUNet(nn.Module):\\n    def __init__(self, pretrained=True):\\n        super().__init__()\\n        encoder = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)\\n        self.stem = nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu)\\n        self.pool, self.layer1, self.layer2, self.layer3, self.layer4 = encoder.maxpool, encoder.layer1, encoder.layer2, encoder.layer3, encoder.layer4\\n        self.up4, self.dec4 = nn.ConvTranspose2d(512, 256, 2, 2), ConvBlock(512, 256)\\n        self.up3, self.dec3 = nn.ConvTranspose2d(256, 128, 2, 2), ConvBlock(256, 128)\\n        self.up2, self.dec2 = nn.ConvTranspose2d(128, 64, 2, 2), ConvBlock(128, 64)\\n        self.up1, self.dec1 = nn.ConvTranspose2d(64, 32, 2, 2), ConvBlock(96, 32)\\n        self.up0, self.head = nn.ConvTranspose2d(32, 16, 2, 2), nn.Conv2d(16, 1, 1)\\n        self.register_buffer(\\"mean\\", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))\\n        self.register_buffer(\\"std\\", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))\\n\\n    def forward(self, image):\\n        image = (image - self.mean) / self.std\\n        x0 = self.stem(image)\\n        x1 = self.layer1(self.pool(x0))\\n        x2 = self.layer2(x1)\\n        x3 = self.layer3(x2)\\n        x4 = self.layer4(x3)\\n        x = self.dec4(torch.cat([self.up4(x4), x3], 1))\\n        x = self.dec3(torch.cat([self.up3(x), x2], 1))\\n        x = self.dec2(torch.cat([self.up2(x), x1], 1))\\n        x = self.dec1(torch.cat([self.up1(x), x0], 1))\\n        return self.head(self.up0(x))\\n\\ndef soft_dice_loss(logits, targets):\\n    probabilities = logits.sigmoid()\\n    intersection = (probabilities * targets).sum((1, 2, 3))\\n    denominator = probabilities.sum((1, 2, 3)) + targets.sum((1, 2, 3))\\n    return (1 - (2 * intersection + 1) / (denominator + 1)).mean()\\n\\ndef train_refiner(model, train_loader, epochs, output_path, val_loader=None):\\n    device = torch.device(\\"cuda:0\\")\\n    model.to(device)\\n    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)\\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)\\n    scaler = torch.amp.GradScaler(\\"cuda\\")\\n    history, best_dice = [], -1.0\\n\\n    for epoch in range(1, epochs + 1):\\n        model.train()\\n        loss_sum = bce_sum = dice_sum = samples = 0\\n\\n        for images, masks in tqdm(train_loader, desc=f\\"Refiner epoch {epoch:02d}/{epochs}\\", leave=False):\\n            images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)\\n            optimizer.zero_grad(set_to_none=True)\\n\\n            with torch.amp.autocast(\\"cuda\\", dtype=torch.float16):\\n                logits = model(images)\\n                bce = nn.functional.binary_cross_entropy_with_logits(logits, masks)\\n                dice_loss = soft_dice_loss(logits, masks)\\n                loss = 0.5 * bce + 0.5 * dice_loss\\n\\n            scaler.scale(loss).backward()\\n            scaler.step(optimizer)\\n            scaler.update()\\n            batch = len(images)\\n            loss_sum += loss.item() * batch\\n            bce_sum += bce.item() * batch\\n            dice_sum += dice_loss.item() * batch\\n            samples += batch\\n\\n        val_dice = np.nan\\n        if val_loader is not None:\\n            model.eval()\\n            dice_total = val_samples = 0\\n\\n            with torch.inference_mode():\\n                for images, masks in tqdm(val_loader, desc=\\"Refiner validation\\", leave=False):\\n                    masks = masks.to(device, non_blocking=True).bool()\\n                    predictions = model(images.to(device, non_blocking=True)).sigmoid() >= 0.65\\n                    intersection = (predictions & masks).sum((1, 2, 3)).float()\\n                    denominator = predictions.sum((1, 2, 3)) + masks.sum((1, 2, 3))\\n                    dice_total += ((2 * intersection + 1e-7) / (denominator + 1e-7)).sum().item()\\n                    val_samples += len(images)\\n\\n            val_dice = dice_total / val_samples\\n\\n            if val_dice > best_dice:\\n                best_dice = val_dice\\n                torch.save({\\"epoch\\": epoch, \\"model\\": model.state_dict()}, output_path)\\n\\n        scheduler.step()\\n        row = {\\"epoch\\": epoch, \\"loss\\": loss_sum / samples, \\"bce\\": bce_sum / samples, \\"dice_loss\\": dice_sum / samples, \\"val_dice\\": val_dice}\\n        history.append(row)\\n        metric_text = f\\" | Val Dice {val_dice:.4f}\\" if val_loader is not None else \\"\\"\\n        print(f\\"Epoch {epoch:02d}/{epochs} | Loss {row[\'loss\']:.4f} | BCE {row[\'bce\']:.4f} | Dice loss {row[\'dice_loss\']:.4f}{metric_text}\\")\\n\\n    if val_loader is None:\\n        torch.save({\\"epoch\\": epochs, \\"model\\": model.state_dict()}, output_path)\\n\\n    return pd.DataFrame(history)\\n\\ntrain_crop_dataset = CropDataset(train_crop_images, train_crop_masks, augment=True)\\nval_crop_dataset = CropDataset(val_crop_images, val_crop_masks, augment=False)\\ntrain_crop_loader = DataLoader(train_crop_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)\\nval_crop_loader = DataLoader(val_crop_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)\\n\\ncrop_model = CropUNet(pretrained=True)\\ncrop_history = train_refiner(crop_model, train_crop_loader, 8, WORK_DIR / \\"best_refiner.pt\\", val_crop_loader)\\ncrop_history.to_csv(WORK_DIR / \\"refiner_history.csv\\", index=False)\\ndisplay(crop_history.round(4))\\n\\ndel crop_model\\ntorch.cuda.empty_cache()\\n# Four-view flip TTA, selected on held-out official TRAIN observations.\\ndef predict_refiner_probabilities(model, images, use_tta=False):\\n    if not use_tta:\\n        return model(images).sigmoid()[:,0]\\n    logits=[]\\n    for flip_y,flip_x in ((False,False),(False,True),(True,False),(True,True)):\\n        dims=tuple(d for enabled,d in ((flip_y,-2),(flip_x,-1)) if enabled)\\n        transformed=torch.flip(images,dims) if dims else images\\n        result=model(transformed).float()\\n        if dims: result=torch.flip(result,dims)\\n        logits.append(result)\\n    return torch.stack(logits).mean(0).sigmoid()[:,0]\\n","metadata":{},"outputs":[],"execution_count":null},{"id":"5fc410c1-9f83-4610-8050-43aa17118031","cell_type":"code","source":"from collections import defaultdict\\n\\ndef merge_overlapping_rles(rles):\\n    if len(rles) < 2:\\n        return rles\\n\\n    adjacency = mask_utils.iou(rles, rles, [0] * len(rles)) > 0\\n    groups, seen = [], set()\\n\\n    for start in range(len(rles)):\\n        if start in seen:\\n            continue\\n\\n        stack, group = [start], []\\n        while stack:\\n            index = stack.pop()\\n\\n            if index in seen:\\n                continue\\n\\n            seen.add(index); group.append(index)\\n            stack.extend(np.flatnonzero(adjacency[index]).tolist())\\n\\n        groups.append(group)\\n\\n    return [mask_utils.merge([rles[index] for index in group]) for group in groups]\\n\\nwith open(WORK_DIR / \\"best_detector_predictions_1024.pkl\\", \\"rb\\") as file:\\n    best_detector_predictions = pickle.load(file)\\n\\nselected_refs_2048, all_refs_2048 = build_reference_maps(val_rows, 2048)\\nfile_by_observation = manifest.set_index(\\"observation_id\\")[\\"file_name\\"].to_dict()\\nbaseline_predictions_2048, tasks = {}, []\\n\\nfor observation_id, rles in tqdm(best_detector_predictions.items(), desc=\\"Preparing native-resolution crops\\"):\\n    image = cv2.imread(str(TRAIN_DIR / file_by_observation[observation_id]), cv2.IMREAD_GRAYSCALE)\\n    height, width = image.shape\\n    baseline_predictions_2048[observation_id] = []\\n\\n    for rle in rles:\\n        mask = mask_utils.decode(rle).astype(np.uint8)\\n        full_mask = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST).astype(bool)\\n        full_rle = encode_mask(full_mask)\\n        baseline_predictions_2048[observation_id].append(full_rle)\\n        x0, y0, x1, y1 = square_bounds(full_mask)\\n        crop = cv2.resize(image[y0:y1, x0:x1], (256, 256), interpolation=cv2.INTER_LINEAR)\\n        tasks.append({\\"observation_id\\": observation_id, \\"bounds\\": (x0, y0, x1, y1), \\"shape\\": (height, width), \\"crop\\": crop, \\"fallback\\": full_rle})\\n\\nrefiner = CropUNet(pretrained=False).to(\\"cuda:0\\")\\nrefiner_checkpoint = torch.load(WORK_DIR / \\"best_refiner.pt\\", map_location=\\"cuda:0\\", weights_only=False)\\nrefiner.load_state_dict(refiner_checkpoint[\\"model\\"])\\nrefiner.eval()\\n\\nrefined_unmerged = {\'plain\': defaultdict(list), \'flip4\': defaultdict(list)}\\nrefiner_threshold = 0.85\\n\\nwith torch.inference_mode():\\n    for start in tqdm(range(0, len(tasks), 32), desc=\\"Refining detector masks\\"):\\n        batch_tasks = tasks[start:start + 32]\\n        images = torch.from_numpy(np.stack([task[\\"crop\\"] for task in batch_tasks])).float().unsqueeze(1).repeat(1, 3, 1, 1).to(\\"cuda:0\\") / 255\\n        probs_by_mode={\\n            mode: predict_refiner_probabilities(refiner,images,use_tta=(mode==\'flip4\')).cpu().numpy()\\n            for mode in (\'plain\',\'flip4\')\\n        }\\n        for task_index,task in enumerate(batch_tasks):\\n            for mode in (\'plain\',\'flip4\'):\\n                probability=probs_by_mode[mode][task_index]\\n                x0,y0,x1,y1=task[\'bounds\']\\n                height,width=task[\'shape\']\\n                crop_mask=cv2.resize(probability,(x1-x0,y1-y0),interpolation=cv2.INTER_LINEAR)>=refiner_threshold\\n                if np.any(crop_mask):\\n                    full_mask=np.zeros((height,width),dtype=np.uint8)\\n                    full_mask[y0:y1,x0:x1]=crop_mask\\n                    refined_unmerged[mode][task[\'observation_id\']].append(encode_mask(full_mask))\\n                else:\\n                    refined_unmerged[mode][task[\'observation_id\']].append(task[\'fallback\'])\\n\\nrefined_by_mode={\\n    mode:{obs:merge_overlapping_rles(predictions.get(obs,[]))\\n          for obs in best_detector_predictions}\\n    for mode,predictions in refined_unmerged.items()\\n}\\nrefined_predictions_2048=refined_by_mode[\'plain\']\\ncomparison=[]\\nfor name,predictions in [(\'Detector\',baseline_predictions_2048),\\n                         (\'Detector + refiner\',refined_by_mode[\'plain\']),\\n                         (\'Detector + refiner + flip4\',refined_by_mode[\'flip4\'])]:\\n    selected_metrics = evaluate_prediction_map(predictions, selected_refs_2048)\\n    all_metrics = evaluate_prediction_map(predictions, all_refs_2048)\\n    comparison.append({\\"strategy\\": name, \\"selected_dice\\": selected_metrics[\\"mean_dice\\"], \\"all_dice\\": all_metrics[\\"mean_dice\\"], \\"all_union_dice\\": all_metrics[\\"union_dice\\"], \\"all_f1_50\\": all_metrics[\\"f1_50\\"], \\"count_error\\": all_metrics[\\"count_error\\"], \\"predictions\\": all_metrics[\\"predictions\\"]})\\n\\ncomparison = pd.DataFrame(comparison).set_index(\\"strategy\\")\\nwith open(WORK_DIR / \\"refined_validation_predictions_2048.pkl\\", \\"wb\\") as file:\\n    pickle.dump(refined_predictions_2048, file)\\n\\ndisplay(comparison.round(3))\\n\\ndel refiner\\ntorch.cuda.empty_cache()","metadata":{},"outputs":[],"execution_count":null},{"id":"61e1797e-41c5-4a10-97d8-9b3b2714339f","cell_type":"code","source":"detector_history = pd.read_csv(WORK_DIR / \\"validation_detector_history.csv\\")\\nbest_detector_epoch = max(6, int(detector_history.loc[detector_history[\\"mean_dice\\"].idxmax(), \\"epoch\\"]))\\nprint(f\\"Final detector epochs: {best_detector_epoch}\\")","metadata":{},"outputs":[],"execution_count":null},{"id":"0ce3fc75-a022-40fa-9a01-70b1737e71f0","cell_type":"code","source":"!torchrun --standalone --nproc_per_node=2 /kaggle/working/solar_filaments_final/train_detector.py --mode final --epochs {best_detector_epoch}","metadata":{},"outputs":[],"execution_count":null},{"id":"eac02e99-0497-4b3a-9b06-fd276b5f4521","cell_type":"code","source":"best_refiner_epoch = max(5, int(crop_history.loc[crop_history[\\"val_dice\\"].idxmax(), \\"epoch\\"]))\\nbest_strategy=str(comparison[\'all_dice\'].idxmax())\\nuse_refiner=best_strategy!=\'Detector\'\\nuse_refiner_tta=best_strategy==\'Detector + refiner + flip4\'\\nprint(f\\"Use refiner: {use_refiner} | Final refiner epochs: {best_refiner_epoch}\\")\\n\\nif use_refiner:\\n    final_train_crops = ConcatDataset([CropDataset(train_crop_images, train_crop_masks, augment=True), CropDataset(val_crop_images, val_crop_masks, augment=True)])\\n    final_crop_loader = DataLoader(final_train_crops, batch_size=16, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)\\n    final_refiner = CropUNet(pretrained=True)\\n    final_refiner_history = train_refiner(final_refiner, final_crop_loader, best_refiner_epoch, WORK_DIR / \\"final_refiner.pt\\")\\n    final_refiner_history.to_csv(WORK_DIR / \\"final_refiner_history.csv\\", index=False)\\n    del final_refiner\\n    torch.cuda.empty_cache()\\n\\nwith open(WORK_DIR / \\"detector_thresholds.json\\") as file:\\n    detector_thresholds = json.load(file)\\n\\nfinal_configuration = {\\"detector_epochs\\": best_detector_epoch, \\"score_threshold\\": detector_thresholds[\\"score_threshold\\"], \\"mask_threshold\\": detector_thresholds[\\"mask_threshold\\"], \\"overlap_merge\\": True, \\"use_refiner\\": use_refiner, \\"refiner_flip4\\": use_refiner_tta, \\"refiner_epochs\\": best_refiner_epoch if use_refiner else None, \\"refiner_threshold\\": 0.85 if use_refiner else None}\\n\\nwith open(WORK_DIR / \\"final_configuration.json\\", \\"w\\") as file:\\n    json.dump(final_configuration, file, indent=2)\\n\\nprint(final_configuration)","metadata":{},"outputs":[],"execution_count":null},{"id":"6f0c29e8-f394-4c09-9792-40c0a9a731a4","cell_type":"markdown","source":"# 4. Submission","metadata":{}},{"id":"fc6362f7-02fc-4fe3-8e6a-7df15a357935","cell_type":"code","source":"import json, pickle\\nfrom pathlib import Path\\nimport cv2, numpy as np, pandas as pd, torch\\nfrom pycocotools import mask as mask_utils\\nfrom tqdm.auto import tqdm\\n\\nwith open(WORK_DIR / \\"final_configuration.json\\") as file:\\n    final_configuration = json.load(file)\\n\\ndevice = torch.device(\\"cuda:0\\")\\nfinal_detector = detector.build_model(pretrained=False).to(device)\\ndetector_checkpoint = torch.load(WORK_DIR / \\"final_detector.pt\\", map_location=device, weights_only=False)\\nfinal_detector.load_state_dict(detector_checkpoint[\\"model\\"])\\nfinal_detector.eval()\\n\\nfinal_refiner = None\\nif final_configuration[\\"use_refiner\\"]:\\n    final_refiner = CropUNet(pretrained=False).to(device)\\n    refiner_checkpoint = torch.load(WORK_DIR / \\"final_refiner.pt\\", map_location=device, weights_only=False)\\n    final_refiner.load_state_dict(refiner_checkpoint[\\"model\\"])\\n    final_refiner.eval()\\n\\nscore_threshold = final_configuration[\\"score_threshold\\"]\\nmask_threshold = final_configuration[\\"mask_threshold\\"]\\nrefiner_threshold = final_configuration.get(\\"refiner_threshold\\", 0.85)\\ntest_files = sorted(path for path in TEST_DIR.iterdir() if path.suffix.lower() in {\\".jpeg\\", \\".jpg\\", \\".png\\"})\\ntest_predictions = {}\\n\\nwith torch.inference_mode():\\n    for image_path in tqdm(test_files, desc=\\"Predicting test images\\"):\\n        observation_id = image_path.stem\\n        image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)\\n        height, width = image.shape\\n        resized = cv2.resize(image, (1024, 1024), interpolation=cv2.INTER_AREA).astype(np.float32) / 255\\n        tensor = torch.from_numpy(resized).unsqueeze(0).repeat(3, 1, 1).to(device)\\n\\n        output = final_detector([tensor])[0]\\n        keep = output[\\"scores\\"] >= score_threshold\\n        detector_masks = (output[\\"masks\\"][keep, 0] >= mask_threshold).byte().cpu().numpy().astype(bool)\\n        detector_masks = detector.merge_overlaps(detector_masks)\\n\\n        full_detector_masks, crops, bounds = [], [], []\\n        for mask in detector_masks:\\n            full_mask = cv2.resize(mask.astype(np.uint8), (width, height), interpolation=cv2.INTER_NEAREST).astype(bool)\\n            x0, y0, x1, y1 = square_bounds(full_mask)\\n            full_detector_masks.append(full_mask)\\n            bounds.append((x0, y0, x1, y1))\\n            crops.append(cv2.resize(image[y0:y1, x0:x1], (256, 256), interpolation=cv2.INTER_LINEAR))\\n\\n        prediction_rles = []\\n        if final_refiner is not None and crops:\\n            crop_tensor = torch.from_numpy(np.stack(crops)).float().unsqueeze(1).repeat(1, 3, 1, 1).to(device) / 255\\n            probabilities = predict_refiner_probabilities(\\n                final_refiner,crop_tensor,\\n                use_tta=final_configuration.get(\'refiner_flip4\',False)\\n            ).cpu().numpy()\\n\\n            for full_detector_mask, probability, (x0, y0, x1, y1) in zip(full_detector_masks, probabilities, bounds):\\n                crop_mask = cv2.resize(probability, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR) >= refiner_threshold\\n\\n                if np.any(crop_mask):\\n                    full_mask = np.zeros((height, width), dtype=np.uint8)\\n                    full_mask[y0:y1, x0:x1] = crop_mask\\n                    prediction_rles.append(encode_mask(full_mask))\\n                else:\\n                    prediction_rles.append(encode_mask(full_detector_mask))\\n        else:\\n            prediction_rles = [encode_mask(mask) for mask in full_detector_masks]\\n\\n        prediction_rles = merge_overlapping_rles(prediction_rles)\\n        prediction_rles.sort(key=lambda rle: (float(mask_utils.toBbox(rle)[1]), float(mask_utils.toBbox(rle)[0])))\\n        test_predictions[observation_id] = prediction_rles\\n\\nwith open(WORK_DIR / \\"test_predictions_2048.pkl\\", \\"wb\\") as file:\\n    pickle.dump(test_predictions, file)\\n\\ndel final_detector, final_refiner\\ntorch.cuda.empty_cache()","metadata":{},"outputs":[],"execution_count":null},{"id":"7be0ef8e-5689-401a-b40c-7beb2eda4f53","cell_type":"code","source":"submission_rows = []\\nfor observation_id, rles in test_predictions.items():\\n    for filament_number, rle in enumerate(rles, 1):\\n        counts = rle[\\"counts\\"].decode(\\"ascii\\") if isinstance(rle[\\"counts\\"], bytes) else rle[\\"counts\\"]\\n        submission_rows.append({\\"filament_id\\": f\\"{observation_id}_{filament_number}\\", \\"segmentation_rle\\": counts})\\n\\nsubmission = pd.DataFrame(submission_rows, columns=[\\"filament_id\\", \\"segmentation_rle\\"])\\nsubmission_path = Path(\\"/kaggle/working/submission.csv\\")\\nsubmission.to_csv(submission_path, index=False)\\n\\nprediction_counts = pd.Series({observation_id: len(rles) for observation_id, rles in test_predictions.items()}, name=\\"predicted_filaments\\")\\nareas = np.array([float(mask_utils.area(rle)) for rles in test_predictions.values() for rle in rles])\\ntest_ids = {path.stem for path in test_files}\\n\\nprint(f\\"Submission:               {submission_path}\\")\\nprint(f\\"Submission rows:          {len(submission):,}\\")\\nprint(f\\"Test images represented:  {len(test_predictions)} / {len(test_files)}\\")\\nprint(f\\"Unique filament IDs:      {submission.filament_id.is_unique}\\")\\nprint(f\\"Prediction count/image:   mean={prediction_counts.mean():.2f}, median={prediction_counts.median():.1f}, min={prediction_counts.min()}, max={prediction_counts.max()}\\")\\nprint(f\\"Predicted mask area:       median={np.median(areas):.0f}, min={areas.min():.0f}, max={areas.max():.0f}\\")\\nprint(f\\"Images with no prediction: {(prediction_counts == 0).sum()}\\")\\nprint(f\\"All test IDs covered:      {set(test_predictions) == test_ids}\\")\\nprint(f\\"File size:                 {submission_path.stat().st_size / 2**20:.2f} MB\\")\\ndisplay(submission.head())","metadata":{},"outputs":[],"execution_count":null},{"id":"457fd555-fe30-457d-a508-d673d3796a6c","cell_type":"markdown","source":"## Submission gate\\nValidate 180 images and RLE integrity; test labels are never read.","metadata":{}},{"id":"f968b517-19a7-4d4c-a9a8-16b46aef5104","cell_type":"code","source":"# Submission integrity; test images remain inference-only.\\nfrom pycocotools import mask as mask_utils\\nassert len(test_files)==180, f\'Expected 180 test images, found {len(test_files)}\'\\nassert set(test_predictions)=={p.stem for p in test_files}\\nassert list(submission.columns)==[\'filament_id\',\'segmentation_rle\']\\nassert submission.filament_id.is_unique and submission.segmentation_rle.notna().all()\\nassert len(submission)>0, \'Empty submission rejected\'\\nids=set(submission.filament_id)\\nfor observation,masks in test_predictions.items():\\n    for j,rle in enumerate(masks,1):\\n        assert tuple(rle[\'size\'])==(2048,2048)\\n        assert int(mask_utils.area(rle))>0\\n        assert f\'{observation}_{j}\' in ids\\nprint(\'Submission gate passed:\',len(test_files),\'images,\',len(submission),\'RLE instances.\')\\nprint(\'Official public score is unknown until submission.\')\\n","metadata":{},"outputs":[],"execution_count":null}]}'
PACKAGE=WORK/'package'
for name,source in sources.items():
    path=PACKAGE/name;path.parent.mkdir(parents=True,exist_ok=True);path.write_text(source)
source_path=WORK/'archived_v5_training_v3.ipynb';source_path.write_text(baseline)
sys.path.insert(0,str(PACKAGE))
from solar.v6_resolution_audit import run
INPUT=Path('/kaggle/input')
def one(name):
    files=list(INPUT.rglob(name))
    if len(files)!=1:raise RuntimeError(f'Expected one {name}, found {len(files)}')
    return files[0]
result=run(one('MAGFiLO_1.0_Annotations_kaggle2026_train.json'),one('training_manifest.csv'),
           one('best_detector_predictions_1024.pkl'),one('refined_validation_predictions_2048.pkl'),
           one('best_detector.pt'),one('best_refiner.pt'),source_path,WORK,seconds=1000)
print('Inference-only local validation complete. No training, test inference, or official submission.')
print(json.dumps({'selected':result['selected'],'confirmation_gain_supported':result['confirmation_gain_supported']},indent=2))
